# 🎵 Advanced Multi-Modal VAE Music Clustering — Combined (Medium + Hard Task)
### Conv2D-VAE · Beta-VAE · CVAE · Conv1D-VAE · Autoencoder · MultiModalVAE · HybridConvVAE

| Component | Details |
|-----------|---------|
| 🏗️ **MLP-VAE** | Standard fully-connected VAE baseline |
| 🌀 **Conv2D-VAE** | Conv2D encoder on delta-stacked MFCC (3×N_MFCC×T) |
| 🔀 **HybridConvVAE** | End-to-end Conv2D + Lyric feature-level fusion |
| 📝 **Hybrid-MLP-VAE** | Audio flat ‖ Lyrics → MLP-VAE |
| 🎛️ **Beta-VAE** | Disentangled latent (β-sweep per dataset) |
| 🏷️ **CVAE** | Genre-conditional VAE |
| 〰️ **Conv1D-VAE** | Conv1D encoder on flat feature vector |
| 🔲 **Autoencoder** | Deterministic AE baseline |
| 🎼 **MultiModalVAE** | Joint audio+lyric encoder |
| 🔢 **4 Clustering Algos** | K-Means · Agglom-Ward · Agglom-Complete · DBSCAN |
| 📊 **6 Metrics** | Silhouette · Davies-Bouldin · Calinski-Harabasz · ARI · NMI · Purity |
| 🇧🇩 **Datasets** | GTZAN · BanglaGITI · BMGCD (real audio only) |


## ⚙️ Step 1: Install Dependencies

In [ ]:
!pip install -q umap-learn scikit-learn matplotlib seaborn tqdm requests
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q yt-dlp librosa soundfile audioread
!pip install -q lyricsgenius beautifulsoup4
!pip install -q gdown kaggle
!apt-get install -q -y ffmpeg
print('✅ All packages installed!')


## 📦 Step 2: Imports & Global Config

In [ ]:
import os, zipfile, tarfile, warnings, re, copy, time, json, subprocess, glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from collections import Counter, defaultdict
import requests
from bs4 import BeautifulSoup
import lyricsgenius
import librosa
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler, LabelEncoder, normalize
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score,
    davies_bouldin_score, adjusted_rand_score,
    normalized_mutual_info_score,
)
import umap

warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Core hyperparameters ───────────────────────────────────────────────────────
DEVICE               = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_SAMPLES          = 10_000
LATENT_DIM           = 32
HIDDEN_DIMS          = (256, 128, 64)
CONV_CHANNELS        = (32, 64, 128)
EPOCHS               = 100
EARLY_STOP_PATIENCE  = 10
LR                   = 1e-3
BETA                 = 1.0
BETA_VAE_B           = 4.0
BETA_VALUES          = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0]
BATCH_SIZE           = 256
LYRIC_DIM            = 128
FUSION_DIM           = 256
N_PER_GENRE          = 20
MIN_SAMPLES_FOR_METRICS = 30
KMEANS_NINIT         = 20

# ── Spectrogram config (NB2: 3×N_MFCC rows with delta+delta2) ─────────────────
N_MFCC        = 20
TIME_FRAMES   = 128
N_MFCC_ROWS   = 3 * N_MFCC   # 60 (MFCC + Δ + Δ²)
MFCC_2D_DIM   = N_MFCC_ROWS * TIME_FRAMES   # 7680
AUDIO_FEAT_DIM = 65   # 65-dim flat feature vector

# ── Lyrics config ──────────────────────────────────────────────────────────────
GENIUS_TOKEN = os.getenv('GENIUS_TOKEN', 'P_idh3O0SAtctPm4mwZZObR0jkagRyOcTYcTAO89DYHfo5auQF6o0AINS63JUx0O')
LYRICS_CACHE = '/content/lyrics_cache'
os.makedirs(LYRICS_CACHE, exist_ok=True)

# ── Dataset directories ────────────────────────────────────────────────────────
GTZAN_DIR      = '/content/gtzan'
BANGLAGITI_DIR = '/content/banglagiti'
BMGCD_DIR      = '/content/bmgcd'
BANGLA_YT_DIR  = '/content/bangla_audio'
OUTPUT_DIR     = '/content/vae_combined_outputs'
for d in [GTZAN_DIR, BANGLAGITI_DIR, BMGCD_DIR, BANGLA_YT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── yt-dlp fallback queries ────────────────────────────────────────────────────
BANGLA_QUERIES_YT = {
    'Rabindra_Sangeet': 'rabindra sangeet full song playlist',
    'Baul':             'baul song bangla authentic',
    'Folk':             'bangla folk song lok geeti traditional',
    'Modern_Pop':       'bangla modern song adhunik gaan',
    'Classical':        'bangla classical music raga',
}
N_BANGLA_PER_GENRE = 20

BMGCD_QUERIES_YT = {
    'Adhunik':   'bangla adhunik gaan modern',
    'Baul':      'baul gaan bengali mystical folk',
    'Classical': 'bangla shashtriya sangeet raga',
    'Folk':      'bangla palli geeti folk rural',
    'Rabindra':  'rabindra sangeet tagore',
}

# ── Model display labels & colors ──────────────────────────────────────────────
MODEL_LABELS = {
    'mlp':      'MLP-VAE',
    'conv':     'Conv2D-VAE',
    'hyb_conv': 'Hybrid-Conv-VAE',
    'hyb_mlp':  'Hybrid-MLP-VAE',
    'beta':     'Beta-VAE',
    'cvae':     'CVAE',
    'conv1d':   'Conv1D-VAE',
    'ae':       'Autoencoder',
    'mm':       'MultiModalVAE',
    'pca':      'PCA',
    'raw':      'Raw-Spectral',
}

COLORS_M = {
    'MLP-VAE':         '#1565C0',
    'Conv2D-VAE':      '#6A1B9A',
    'Hybrid-Conv-VAE': '#2E7D32',
    'Hybrid-MLP-VAE':  '#E65100',
    'Beta-VAE':        '#AD1457',
    'CVAE':            '#00838F',
    'Conv1D-VAE':      '#558B2F',
    'Autoencoder':     '#FF8F00',
    'MultiModalVAE':   '#00695C',
    'PCA':             '#B71C1C',
    'Raw-Spectral':    '#546E7A',
}

LANG_MK  = {'English': 'o', 'Bangla': '^'}
LANG_COL = {'English': '#1565C0', 'Bangla': '#C62828'}

# Subsets for plot filtering
Z_KEYS_MED = ['mlp', 'conv', 'hyb_conv', 'hyb_mlp', 'pca']
Z_KEYS_ALL = ['mlp', 'conv', 'hyb_conv', 'hyb_mlp',
              'beta', 'cvae', 'conv1d', 'ae', 'mm', 'pca', 'raw']

print('✅ Config ready | Device:', DEVICE)
print(f'   N_MFCC_ROWS={N_MFCC_ROWS}  TIME_FRAMES={TIME_FRAMES}  MFCC_2D_DIM={MFCC_2D_DIM}')
print(f'   AUDIO_FEAT_DIM={AUDIO_FEAT_DIM}  LYRIC_DIM={LYRIC_DIM}  LATENT_DIM={LATENT_DIM}')


## 🔧 Step 3: Conv2D Normalization Helper

In [ ]:
def normalize_for_conv2d(X_flat, n_rows=N_MFCC_ROWS, time_frames=TIME_FRAMES):
    """
    Per-coefficient (row) standardization for Conv2D input.
    Input  : (N, n_rows * time_frames) flat
    Output : (N, n_rows * time_frames) per-row normalized
    Do NOT pass StandardScaler output — causes double normalization.
    """
    X_2d = X_flat.reshape(-1, n_rows, time_frames).copy()   # (N, 60, 128)
    mean = X_2d.mean(axis=(0, 2), keepdims=True)            # (1, 60, 1)
    std  = X_2d.std(axis=(0, 2),  keepdims=True) + 1e-8
    X_2d = (X_2d - mean) / std
    return X_2d.reshape(-1, n_rows * time_frames).astype(np.float32)


def align_for_conv2d(X_sc, target=None):
    """Pad or crop flat array to exactly N_MFCC_ROWS * TIME_FRAMES."""
    expected = target or (N_MFCC_ROWS * TIME_FRAMES)
    current  = X_sc.shape[1]
    if current == expected:
        return X_sc
    if current < expected:
        return np.pad(X_sc, ((0, 0), (0, expected - current)))
    return X_sc[:, :expected]


print('✅ normalize_for_conv2d() and align_for_conv2d() ready.')
print(f'   Conv2D input shape: (N, {N_MFCC_ROWS}, {TIME_FRAMES}) = (N, {MFCC_2D_DIM}) flat')


## 🏗️ Step 4: All Model Architectures

| Model | Type | Input | Key Feature |
|-------|------|-------|-------------|
| `MLPVAE` | Probabilistic | flat (N, D) | Standard ELBO |
| `BetaVAE` | Probabilistic | flat (N, D) | β-weighted KL for disentanglement |
| `CVAE` | Conditional | flat (N, D) + onehot | Genre label concatenated |
| `ConvVAE` | Probabilistic | flat (N, D) | Conv1D encoder |
| `Autoencoder` | Deterministic | flat (N, D) | No KL baseline |
| `MultiModalVAE` | Probabilistic | audio + lyric | Joint audio+lyric encoder |
| `Conv2DEncoder/Decoder` | — | (N,1,60,128) | Conv2D spectrogram encoder |
| `Conv2DVAE` | Probabilistic | (N,7680) flat | Full Conv2D-VAE |
| `HybridConvVAE` | Probabilistic | (N,7808) | Conv2D features fused with lyrics |


In [ ]:
# ── Shared helpers ─────────────────────────────────────────────────────────────
def make_mlp(dims, activation=nn.LeakyReLU, dropout=0.2, bn=True):
    """Build MLP with BN + activation on hidden layers only."""
    layers, prev = [], dims[0]
    for i, d in enumerate(dims[1:], 1):
        is_last = (i == len(dims) - 1)
        layers.append(nn.Linear(prev, d))
        if not is_last:
            if bn:
                layers.append(nn.BatchNorm1d(d))
            layers.append(
                activation(0.2) if activation == nn.LeakyReLU else activation()
            )
            if dropout:
                layers.append(nn.Dropout(dropout))
        prev = d
    return nn.Sequential(*layers)


def vae_loss_fn(recon, x, mu, lv, beta=1.0):
    """ELBO: reconstruction (MSE) + β * KL divergence, averaged per sample."""
    x_flat = x.view(x.size(0), -1)
    rl = F.mse_loss(recon, x_flat, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp()) / x.size(0)
    return rl + beta * kl, rl.item(), kl.item()


# ── A) MLP-VAE ─────────────────────────────────────────────────────────────────
class MLPVAE(nn.Module):
    def __init__(self, in_dim, z_dim=LATENT_DIM, h=HIDDEN_DIMS):
        super().__init__()
        self.enc_net = make_mlp([in_dim] + list(h))
        self.mu_fc   = nn.Linear(h[-1], z_dim)
        self.lv_fc   = nn.Linear(h[-1], z_dim)
        self.dec_net = make_mlp([z_dim] + list(reversed(h)) + [in_dim])

    def encode(self, x):
        h = self.enc_net(x)
        return self.mu_fc(h), self.lv_fc(h)

    def reparameterize(self, mu, lv):
        lv = torch.clamp(lv, -10, 10)
        return mu + torch.randn_like(mu) * torch.exp(0.5 * lv) if self.training else mu

    def decode(self, z):
        return self.dec_net(z)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = self.reparameterize(mu, lv)
        return self.decode(z), mu, lv, z

    def get_latent(self, x):
        mu, _ = self.encode(x)
        return mu


# ── B) Beta-VAE ────────────────────────────────────────────────────────────────
class BetaVAE(nn.Module):
    """Deeper VAE with tighter lv clamp for stable high-β training."""
    def __init__(self, in_dim, z_dim=LATENT_DIM, beta=4.0, h=(512, 256, 128, 64)):
        super().__init__()
        self.beta   = beta
        self.z_dim  = z_dim
        self.in_dim = in_dim
        self.enc_net = make_mlp([in_dim] + list(h))
        self.mu_fc   = nn.Linear(h[-1], z_dim)
        self.lv_fc   = nn.Linear(h[-1], z_dim)
        self.dec_net = make_mlp([z_dim] + list(reversed(h)) + [in_dim])

    def encode(self, x):
        return self.mu_fc(self.enc_net(x)), self.lv_fc(self.enc_net(x))

    def reparameterize(self, mu, lv):
        lv = torch.clamp(lv, -4, 4)
        return mu + torch.randn_like(mu) * torch.exp(0.5 * lv) if self.training else mu

    def decode(self, z):
        return self.dec_net(z)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = self.reparameterize(mu, lv)
        return self.decode(z), mu, lv, z

    def get_latent(self, x):
        mu, _ = self.encode(x)
        return mu

    @torch.no_grad()
    def disentanglement_score(self, X_sc, batch_size=256):
        self.eval()
        mus = []
        X_t = torch.FloatTensor(X_sc)
        for i in range(0, len(X_t), batch_size):
            bx = X_t[i:i+batch_size].to(next(self.parameters()).device)
            mu, _ = self.encode(bx)
            mus.append(mu.cpu())
        mus = torch.cat(mus, dim=0)
        var_per_dim = mus.var(dim=0)
        p = var_per_dim / (var_per_dim.sum() + 1e-8)
        entropy = -(p * torch.log(p + 1e-8)).sum().item()
        print(f'  Dim variances (top-10): {var_per_dim.topk(min(10, self.z_dim)).values.numpy().round(3)}')
        print(f'  Variance entropy: {entropy:.4f}  (lower = more axis-aligned)')
        return var_per_dim.numpy(), entropy


# ── C) CVAE ────────────────────────────────────────────────────────────────────
class CVAE(nn.Module):
    def __init__(self, in_dim, n_class, z_dim=LATENT_DIM, h=HIDDEN_DIMS):
        super().__init__()
        self.n_class = n_class
        self.enc_net = make_mlp([in_dim + n_class] + list(h))
        self.mu_fc   = nn.Linear(h[-1], z_dim)
        self.lv_fc   = nn.Linear(h[-1], z_dim)
        self.dec_net = make_mlp([z_dim + n_class] + list(reversed(h)) + [in_dim])

    def encode(self, x, c):
        return self.mu_fc(self.enc_net(torch.cat([x, c], dim=1))),                self.lv_fc(self.enc_net(torch.cat([x, c], dim=1)))

    def reparameterize(self, mu, lv):
        lv = torch.clamp(lv, -10, 10)
        return mu + torch.randn_like(mu) * torch.exp(0.5 * lv) if self.training else mu

    def decode(self, z, c):
        return self.dec_net(torch.cat([z, c], dim=1))

    def forward(self, x, c):
        mu, lv = self.encode(x, c)
        z = self.reparameterize(mu, lv)
        return self.decode(z, c), mu, lv, z

    def encode_unconditional(self, x):
        c = torch.zeros(x.size(0), self.n_class, device=x.device)
        return self.encode(x, c)

    def get_latent(self, x):
        mu, _ = self.encode_unconditional(x)
        return mu


# ── D) Conv1D-VAE ──────────────────────────────────────────────────────────────
class ConvVAE(nn.Module):
    def __init__(self, in_dim, z_dim=LATENT_DIM, channels=(32, 64, 128)):
        super().__init__()
        self.in_dim = in_dim
        enc_layers, prev = [], 1
        for ch in channels:
            enc_layers += [nn.Conv1d(prev, ch, 5, stride=2, padding=2),
                           nn.BatchNorm1d(ch), nn.LeakyReLU(0.2)]
            prev = ch
        self.conv_enc = nn.Sequential(*enc_layers)
        with torch.no_grad():
            dummy = self.conv_enc(torch.zeros(1, 1, in_dim))
        raw_flat = dummy.view(1, -1).shape[1]
        self.ch0  = channels[-1]
        self.seq0 = raw_flat // self.ch0
        self.flat = self.seq0 * self.ch0
        self.mu_fc  = nn.Linear(self.flat, z_dim)
        self.lv_fc  = nn.Linear(self.flat, z_dim)
        self.fc_dec = nn.Linear(z_dim, self.flat)
        dec_layers, prev = [], channels[-1]
        for ch in reversed(channels[:-1]):
            dec_layers += [nn.ConvTranspose1d(prev, ch, 4, stride=2, padding=1),
                           nn.BatchNorm1d(ch), nn.LeakyReLU(0.2)]
            prev = ch
        dec_layers.append(nn.ConvTranspose1d(prev, 1, 4, stride=2, padding=1))
        self.conv_dec = nn.Sequential(*dec_layers)

    def encode(self, x):
        h = self.conv_enc(x.unsqueeze(1))
        h = h.view(x.size(0), self.ch0, -1)[:, :, :self.seq0].reshape(x.size(0), -1)
        return self.mu_fc(h), self.lv_fc(h)

    def reparameterize(self, mu, lv):
        lv = torch.clamp(lv, -10, 10)
        return mu + torch.randn_like(mu) * torch.exp(0.5 * lv) if self.training else mu

    def decode(self, z):
        h   = self.fc_dec(z).view(z.size(0), self.ch0, self.seq0)
        out = self.conv_dec(h)
        return F.adaptive_avg_pool1d(out, self.in_dim).squeeze(1)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = self.reparameterize(mu, lv)
        return self.decode(z), mu, lv, z

    def get_latent(self, x):
        mu, _ = self.encode(x)
        return mu


# ── E) Autoencoder (deterministic baseline) ────────────────────────────────────
class Autoencoder(nn.Module):
    def __init__(self, in_dim, z_dim=LATENT_DIM, h=HIDDEN_DIMS):
        super().__init__()
        self.encoder = make_mlp([in_dim] + list(h) + [z_dim])
        self.decoder = make_mlp([z_dim] + list(reversed(h)) + [in_dim])

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def get_latent(self, x):
        return self.encoder(x)

    def enc(self, x):
        return self.encoder(x), None


# ── F) MultiModalVAE ───────────────────────────────────────────────────────────
class MultiModalVAE(nn.Module):
    """Joint audio+lyric encoder. Reconstructs audio only (primary modality)."""
    def __init__(self, audio_dim=AUDIO_FEAT_DIM, lyric_dim=LYRIC_DIM,
                 fusion_dim=FUSION_DIM, z_dim=LATENT_DIM, h=HIDDEN_DIMS):
        super().__init__()
        self.audio_proj = nn.Sequential(
            nn.Linear(audio_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.ReLU()
        )
        self.lyric_proj = nn.Sequential(
            nn.Linear(lyric_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.ReLU()
        )
        self.enc_net = make_mlp([2 * fusion_dim] + list(h))
        self.mu_fc   = nn.Linear(h[-1], z_dim)
        self.lv_fc   = nn.Linear(h[-1], z_dim)
        self.dec_net = make_mlp([z_dim] + list(reversed(h)) + [audio_dim])

    def encode(self, audio, lyric):
        a = self.audio_proj(audio)
        l = self.lyric_proj(lyric)
        h = self.enc_net(torch.cat([a, l], dim=1))
        return self.mu_fc(h), self.lv_fc(h)

    def reparameterize(self, mu, lv):
        lv = torch.clamp(lv, -10, 10)
        return mu + torch.randn_like(mu) * torch.exp(0.5 * lv) if self.training else mu

    def decode(self, z):
        return self.dec_net(z)

    def forward(self, audio, lyric):
        mu, lv = self.encode(audio, lyric)
        z = self.reparameterize(mu, lv)
        return self.decode(z), mu, lv, z

    def get_latent(self, audio, lyric):
        mu, _ = self.encode(audio, lyric)
        return mu


# ── G) Conv2D-VAE ──────────────────────────────────────────────────────────────
class Conv2DEncoder(nn.Module):
    """Input: (B, 1, N_MFCC_ROWS, TIME_FRAMES) = (B, 1, 60, 128)"""
    def __init__(self, n_mfcc=N_MFCC_ROWS, time_frames=TIME_FRAMES,
                 z_dim=LATENT_DIM, channels=CONV_CHANNELS):
        super().__init__()
        layers, in_ch = [], 1
        for out_ch in channels:
            layers += [
                nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2),
            ]
            in_ch = out_ch
        self.conv = nn.Sequential(*layers)
        with torch.no_grad():
            dummy         = torch.zeros(1, 1, n_mfcc, time_frames)
            self.flat_dim = self.conv(dummy).view(1, -1).shape[1]
        self.mu_fc = nn.Linear(self.flat_dim, z_dim)
        self.lv_fc = nn.Linear(self.flat_dim, z_dim)

    def forward(self, x):
        h = self.conv(x).view(x.size(0), -1)
        return self.mu_fc(h), self.lv_fc(h)


class Conv2DDecoder(nn.Module):
    """Input: z (B, z_dim) → reconstructed flat spectrogram (B, N_MFCC_ROWS*TIME_FRAMES)"""
    def __init__(self, z_dim=LATENT_DIM, flat_dim=None,
                 n_mfcc=N_MFCC_ROWS, time_frames=TIME_FRAMES,
                 channels=CONV_CHANNELS):
        super().__init__()
        rev_ch   = list(reversed(channels))
        self.ch0 = rev_ch[0]
        self.h0  = max(1, n_mfcc      // (2 ** len(channels)))
        self.w0  = max(1, time_frames // (2 ** len(channels)))
        self.fc  = nn.Linear(z_dim, self.ch0 * self.h0 * self.w0)
        layers, in_ch = [], rev_ch[0]
        for out_ch in rev_ch[1:]:
            layers += [
                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2),
            ]
            in_ch = out_ch
        layers.append(nn.ConvTranspose2d(in_ch, 1, kernel_size=4, stride=2, padding=1))
        self.deconv   = nn.Sequential(*layers)
        self.out_size = (n_mfcc, time_frames)

    def forward(self, z):
        h   = self.fc(z).view(z.size(0), self.ch0, self.h0, self.w0)
        out = self.deconv(h)
        out = F.adaptive_avg_pool2d(out, self.out_size)
        return out.view(z.size(0), -1)


class Conv2DVAE(nn.Module):
    """
    Full Conv2D-VAE for delta-stacked MFCC spectrograms.
    Accepts flat (B, 7680) or 2D (B, 1, 60, 128) input.
    Input contract: pass normalize_for_conv2d(X_raw_2d).
    """
    def __init__(self, n_mfcc=N_MFCC_ROWS, time_frames=TIME_FRAMES,
                 z_dim=LATENT_DIM, channels=CONV_CHANNELS):
        super().__init__()
        self.n_mfcc      = n_mfcc
        self.time_frames = time_frames
        self.flat_dim    = n_mfcc * time_frames
        self.enc = Conv2DEncoder(n_mfcc, time_frames, z_dim, channels)
        self.dec = Conv2DDecoder(z_dim, self.enc.flat_dim, n_mfcc, time_frames, channels)

    def _to_2d(self, x):
        if x.dim() == 2:
            return x.view(x.size(0), 1, self.n_mfcc, self.time_frames)
        return x

    def reparameterize(self, mu, lv):
        lv = torch.clamp(lv, -10, 10)
        return mu + torch.randn_like(mu) * torch.exp(0.5 * lv) if self.training else mu

    def forward(self, x):
        x2d    = self._to_2d(x)
        mu, lv = self.enc(x2d)
        z      = self.reparameterize(mu, lv)
        return self.dec(z), mu, lv, z

    def get_latent(self, x):
        mu, _ = self.enc(self._to_2d(x))
        return mu


# ── H) HybridConvVAE — end-to-end Conv2D + Lyric feature-level fusion ─────────
#
# Input contract:
#   Passed as single array X_hybrid_conv = np.hstack([X_conv2d, X_lyric_l2])
#   Shape: (N, MFCC_2D_DIM + LYRIC_DIM) = (N, 7808)
#   conv part [:7680] = normalize_for_conv2d output
#   lyric part [7680:] = L2-normalized lyrics from make_multimodal()
#
class HybridConvVAE(nn.Module):
    def __init__(self, lyric_dim=LYRIC_DIM, z_dim=LATENT_DIM,
                 n_mfcc=N_MFCC_ROWS, time_frames=TIME_FRAMES,
                 channels=CONV_CHANNELS, fusion_dim=FUSION_DIM):
        super().__init__()
        self.n_mfcc      = n_mfcc
        self.time_frames = time_frames
        self.conv_dim    = n_mfcc * time_frames   # 7680 = split point
        self.lyric_dim   = lyric_dim

        # Conv2D encoder
        enc_layers, in_ch = [], 1
        for out_ch in channels:
            enc_layers += [
                nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2),
            ]
            in_ch = out_ch
        self.conv_enc = nn.Sequential(*enc_layers)
        with torch.no_grad():
            dummy          = torch.zeros(1, 1, n_mfcc, time_frames)
            self.conv_flat = self.conv_enc(dummy).view(1, -1).shape[1]

        # Lyric projection → same width as conv_flat for balanced fusion
        self.lyric_proj = nn.Sequential(
            nn.Linear(lyric_dim, self.conv_flat),
            nn.LayerNorm(self.conv_flat),
            nn.LeakyReLU(0.2),
        )
        # Fusion: conv_flat + lyric_flat → fusion_dim
        self.fusion = nn.Sequential(
            nn.Linear(self.conv_flat * 2, fusion_dim),
            nn.BatchNorm1d(fusion_dim),
            nn.LeakyReLU(0.2),
        )
        self.mu_fc = nn.Linear(fusion_dim, z_dim)
        self.lv_fc = nn.Linear(fusion_dim, z_dim)

        # Conv2D decoder (mirrors encoder)
        rev_ch      = list(reversed(channels))
        self.ch0    = rev_ch[0]
        self.h0     = max(1, n_mfcc      // (2 ** len(channels)))
        self.w0     = max(1, time_frames // (2 ** len(channels)))
        self.fc_dec = nn.Linear(z_dim, self.ch0 * self.h0 * self.w0)
        dec_layers, in_ch = [], rev_ch[0]
        for out_ch in rev_ch[1:]:
            dec_layers += [
                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2),
            ]
            in_ch = out_ch
        dec_layers.append(nn.ConvTranspose2d(in_ch, 1, kernel_size=4, stride=2, padding=1))
        self.conv_dec = nn.Sequential(*dec_layers)
        self.out_size = (n_mfcc, time_frames)

    def _to_2d(self, x):
        if x.dim() == 2:
            return x.view(x.size(0), 1, self.n_mfcc, self.time_frames)
        return x

    def encode(self, x_conv, x_lyric):
        h_conv  = self.conv_enc(self._to_2d(x_conv)).view(x_conv.size(0), -1)
        h_lyric = self.lyric_proj(x_lyric)
        h_fused = self.fusion(torch.cat([h_conv, h_lyric], dim=1))
        return self.mu_fc(h_fused), self.lv_fc(h_fused)

    def reparameterize(self, mu, lv):
        lv = torch.clamp(lv, -10, 10)
        return mu + torch.randn_like(mu) * torch.exp(0.5 * lv) if self.training else mu

    def decode(self, z):
        h   = self.fc_dec(z).view(z.size(0), self.ch0, self.h0, self.w0)
        out = self.conv_dec(h)
        out = F.adaptive_avg_pool2d(out, self.out_size)
        return out.view(z.size(0), -1)

    def forward(self, x_conv, x_lyric):
        mu, lv = self.encode(x_conv, x_lyric)
        z      = self.reparameterize(mu, lv)
        return self.decode(z), mu, lv, z

    def get_latent(self, x_conv, x_lyric):
        mu, _ = self.encode(x_conv, x_lyric)
        return mu


print('✅ All architectures defined:')
print('   MLPVAE | BetaVAE | CVAE | ConvVAE | Autoencoder')
print('   MultiModalVAE | Conv2DVAE | HybridConvVAE')


## 📝 Step 5: Real Lyrics Pipeline (Genius API + gaanesuno.com)

In [ ]:
# ── Cache helpers ──────────────────────────────────────────────────────────────
def _sanitize(text):
    return re.sub(r'[\\/*?:"<>|]', '', text).strip()[:80]

def _cache_path(genre, language, track_name):
    folder = os.path.join(LYRICS_CACHE, language, genre)
    os.makedirs(folder, exist_ok=True)
    return os.path.join(folder, _sanitize(track_name) + '.txt')

def _load_cached(genre, language, track_name):
    p = _cache_path(genre, language, track_name)
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f:
            return f.read().strip()
    return None

def _save_cached(lyrics, genre, language, track_name):
    with open(_cache_path(genre, language, track_name), 'w', encoding='utf-8') as f:
        f.write(lyrics)

def _is_gtzan_filename(filename):
    base = os.path.splitext(os.path.basename(filename))[0]
    return bool(re.match(r'^[a-z_]+\.\d{5}$', base))

def _parse_title_artist(filename):
    base = re.sub(r'_', ' ', os.path.splitext(os.path.basename(filename))[0])
    if ' - ' in base:
        parts = base.split(' - ', 1)
        return parts[1].strip(), parts[0].strip()
    return base.strip(), None

# ── Genius API (English) ───────────────────────────────────────────────────────
_genius_client = None

def _get_genius():
    global _genius_client
    if _genius_client is None and GENIUS_TOKEN not in ('', 'your_token_here'):
        try:
            _genius_client = lyricsgenius.Genius(GENIUS_TOKEN, timeout=10)
            _genius_client.skip_non_songs  = True
            _genius_client.excluded_terms  = ['(Remix)', '(Live)']
            _genius_client.verbose         = False
        except Exception as e:
            print('Genius init failed:', e)
    return _genius_client

def _clean_genius_lyrics(raw):
    text = re.sub(r'\[.*?\]', '', raw)
    text = re.sub(r'\d+\s*Contributor.*', '', text)
    text = re.sub(r'\d+Embed$', '', text)
    text = re.sub(r'You might also like', '', text)
    return re.sub(r'\s{2,}', ' ', text).strip()

def fetch_english_lyrics(filename, genre, retries=2):
    track_name = os.path.splitext(os.path.basename(filename))[0]
    cached = _load_cached(genre, 'English', track_name)
    if cached:
        return cached
    if _is_gtzan_filename(filename):
        return None
    genius = _get_genius()
    if genius is None:
        return None
    title, artist = _parse_title_artist(filename)
    for attempt in range(retries):
        try:
            song = genius.search_song(title, artist or '')
            if song and song.lyrics:
                lyrics = _clean_genius_lyrics(song.lyrics)
                _save_cached(lyrics, genre, 'English', track_name)
                return lyrics
        except Exception:
            time.sleep(2 ** attempt)
    return None

# ── gaanesuno.com scraper (Bangla) ─────────────────────────────────────────────
HEADERS = {'User-Agent': 'Mozilla/5.0'}

def _scrape_gaanesuno(title, artist=None):
    query_str = (title + ' ' + (artist or '')).strip().replace(' ', '+')
    try:
        r    = requests.get(f'https://www.gaanesuno.com/?s={query_str}',
                            headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        result = (soup.select_one('h2.entry-title a') or
                  soup.select_one('.search-result a') or
                  soup.select_one('article a'))
        if not result:
            return None
        page  = requests.get(result['href'], headers=HEADERS, timeout=10)
        psoup = BeautifulSoup(page.text, 'html.parser')
        content = (psoup.select_one('.entry-content') or
                   psoup.select_one('.lyric-content') or
                   psoup.select_one('article .post-content'))
        if not content:
            return None
        for tag in content(['script', 'style', 'ins', 'figure']):
            tag.decompose()
        lyrics = content.get_text(separator='\n').strip()
        return lyrics if len(lyrics) > 50 else None
    except Exception:
        return None

def fetch_bangla_lyrics(filename, genre, retries=2):
    track_name = os.path.splitext(os.path.basename(filename))[0]
    cached = _load_cached(genre, 'Bangla', track_name)
    if cached:
        return cached
    title, artist = _parse_title_artist(filename)
    for attempt in range(retries):
        try:
            lyrics = _scrape_gaanesuno(title, artist)
            if lyrics:
                _save_cached(lyrics, genre, 'Bangla', track_name)
                return lyrics
        except Exception:
            pass
        time.sleep(2 ** attempt)
    return None

def fetch_lyrics(filename, genre, language):
    if language == 'English':
        return fetch_english_lyrics(filename, genre)
    elif language == 'Bangla':
        return fetch_bangla_lyrics(filename, genre)
    return None

print('✅ Lyrics pipeline ready.')
print('   English: Genius API | Bangla: gaanesuno.com scraper')


## 🎵 Step 6: Multi-Modal Feature Builder

In [ ]:
# Neutral fallback — no genre vocabulary, no label leakage
LYRIC_FALLBACK = 'music sound audio rhythm melody beat instrument song'

def make_multimodal(X_audio_sc, records, lyric_dim=LYRIC_DIM):
    """
    Build multi-modal feature matrices from scaled audio + lyrics.

    Returns
    -------
    X_hybrid   : (N, audio_dim + lyric_dim) — L2(audio) ‖ L2(lyrics)
    has_real   : (N,) bool — True where real lyrics were fetched
    X_lyric_l2 : (N, lyric_dim) — L2-normalized lyrics only
    """
    N = len(records)
    lyric_texts = []
    has_real    = np.zeros(N, dtype=bool)

    for i, rec in enumerate(records):
        fpath    = rec.get('file', '')
        genre    = rec.get('genre', 'other')
        language = rec.get('language', 'English')
        lyric    = fetch_lyrics(fpath, genre, language) if fpath else None
        if lyric:
            lyric_texts.append(lyric)
            has_real[i] = True
        else:
            lyric_texts.append(LYRIC_FALLBACK)

    tfidf   = TfidfVectorizer(max_features=2000, ngram_range=(1, 2),
                              sublinear_tf=True, min_df=1)
    X_tfidf = tfidf.fit_transform(lyric_texts).toarray().astype(np.float32)

    n_comp  = min(lyric_dim, X_tfidf.shape[1], X_tfidf.shape[0] - 1)
    n_comp  = max(n_comp, 2)
    X_lyric = TruncatedSVD(n_components=n_comp, random_state=42).fit_transform(X_tfidf).astype(np.float32)

    if X_lyric.shape[1] < lyric_dim:
        pad     = np.zeros((N, lyric_dim - X_lyric.shape[1]), dtype=np.float32)
        X_lyric = np.hstack([X_lyric, pad])

    audio_l2   = normalize(X_audio_sc, norm='l2')
    X_lyric_l2 = normalize(X_lyric,    norm='l2')
    X_hybrid   = np.hstack([audio_l2, X_lyric_l2]).astype(np.float32)

    real_pct = has_real.mean() * 100
    print(f'  Lyrics     : {has_real.sum()}/{N} real ({real_pct:.1f}%)'
          f' | {N - has_real.sum()} neutral fallback')
    print(f'  X_hybrid   : {X_hybrid.shape}  (L2 audio ‖ L2 lyrics)')
    return X_hybrid, has_real, X_lyric_l2


def make_genre_onehot(y_labels, le):
    """Convert string labels → one-hot using a fitted LabelEncoder."""
    idx = le.transform(y_labels)
    return np.eye(len(le.classes_), dtype=np.float32)[idx]


print('✅ make_multimodal() ready → (X_hybrid, has_real, X_lyric_l2)')


## 🎙️ Step 7: Audio Feature Extraction + Bangla Dataset Builder

In [ ]:
def extract_audio_features(fpath, sr=22050, duration=30, n_mfcc=20):
    """
    65-dim feature vector:
      MFCC mean(20) + std(20) = 40
      Chroma STFT mean        = 12
      Spectral centroid, bandwidth, rolloff, ZCR, RMS = 5
      Tempo = 1 | Spectral contrast mean = 7
    Total = 65
    """
    try:
        y, _ = librosa.load(fpath, sr=sr, duration=duration, mono=True)
        if len(y) < sr * 3:
            return None
        mfcc     = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        chroma   = librosa.feature.chroma_stft(y=y, sr=sr)
        sc       = librosa.feature.spectral_centroid(y=y, sr=sr)
        sb       = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        sr_feat  = librosa.feature.spectral_rolloff(y=y, sr=sr)
        zcr      = librosa.feature.zero_crossing_rate(y)
        rms      = librosa.feature.rms(y=y)
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        feat = np.concatenate([
            mfcc.mean(axis=1), mfcc.std(axis=1),
            chroma.mean(axis=1),
            [sc.mean(), sb.mean(), sr_feat.mean(), zcr.mean(), rms.mean()],
            [float(tempo)],
            contrast.mean(axis=1),
        ]).astype(np.float32)
        return feat
    except Exception:
        return None


def extract_mfcc_2d(fpath, sr=22050, duration=30,
                    n_mfcc=N_MFCC, time_frames=TIME_FRAMES):
    """
    Extract delta-stacked 2D MFCC spectrogram.
    Output: (3 * n_mfcc * time_frames,) flattened = (7680,)
    """
    try:
        y, _ = librosa.load(fpath, sr=sr, duration=duration, mono=True)
        if len(y) < sr * 3:
            return None
        mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        delta  = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        mfcc   = np.vstack([mfcc, delta, delta2])   # (3*n_mfcc, T)
        if mfcc.shape[1] >= time_frames:
            mfcc = mfcc[:, :time_frames]
        else:
            mfcc = np.pad(mfcc, ((0, 0), (0, time_frames - mfcc.shape[1])), mode='constant')
        return mfcc.astype(np.float32).flatten()
    except Exception:
        return None


def download_kaggle_dataset(slug, dest_dir):
    """Download & unzip a Kaggle dataset. Returns dest_dir or None."""
    os.makedirs(dest_dir, exist_ok=True)
    audio_exts = ['.wav', '.mp3', '.flac', '.ogg', '.m4a']
    existing   = []
    for ext in audio_exts:
        existing += glob.glob(f'{dest_dir}/**/*{ext}', recursive=True)
    if existing:
        print(f'  {slug}: {len(existing)} audio files already present.')
        return dest_dir
    kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')
    if not os.path.exists(kaggle_json):
        print(f'  WARNING: kaggle.json not found — cannot download {slug}.')
        return None
    print(f'  Downloading Kaggle: {slug} …')
    try:
        result = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', slug, '-p', dest_dir, '--unzip'],
            capture_output=True, text=True, timeout=600,
        )
        if result.returncode != 0:
            print(f'  kaggle CLI error: {result.stderr[:200]}')
            return None
        found = []
        for ext in audio_exts:
            found += glob.glob(f'{dest_dir}/**/*{ext}', recursive=True)
        print(f'  {slug}: {len(found)} audio files extracted.')
        return dest_dir if found else None
    except Exception as e:
        print(f'  Kaggle download failed: {e}')
        return None


def collect_audio_from_dir(root_dir, min_per_genre=5):
    """Walk root_dir. Genre = immediate parent directory of audio file."""
    audio_exts = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
    records    = []
    root_base  = os.path.basename(os.path.normpath(root_dir))
    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            if os.path.splitext(fname)[1].lower() in audio_exts:
                fpath = os.path.join(dirpath, fname)
                genre = os.path.basename(dirpath)
                if genre in ('.', '', root_base):
                    genre = 'unknown'
                records.append((fpath, genre))
    counts   = Counter(g for _, g in records)
    filtered = [(f, g) for f, g in records if counts[g] >= min_per_genre]
    dropped  = len(records) - len(filtered)
    if dropped:
        print(f'  (dropped {dropped} samples from genres with <{min_per_genre} tracks)')
    return filtered


def download_bangla_yt(genre, query, n=N_BANGLA_PER_GENRE):
    out_dir  = f'{BANGLA_YT_DIR}/{genre}'
    os.makedirs(out_dir, exist_ok=True)
    existing = glob.glob(f'{out_dir}/*.wav') + glob.glob(f'{out_dir}/*.mp3')
    if len(existing) >= n:
        print(f'  {genre}: {len(existing)} files cached.')
        return existing[:n]
    print(f'  yt-dlp {genre}: fetching up to {n} tracks …')
    cmd = [
        'yt-dlp', f'ytsearch{n}:{query}',
        '--extract-audio', '--audio-format', 'wav', '--audio-quality', '5',
        '--no-playlist', '--output', f'{out_dir}/%(id)s.%(ext)s',
        '--no-warnings', '--ignore-errors', '--socket-timeout', '30', '--retries', '3',
    ]
    try:
        result = subprocess.run(cmd, timeout=300, check=False,
                                capture_output=True, text=True)
        if result.returncode != 0 and result.stderr:
            print(f'  yt-dlp stderr: {result.stderr[:200]}')
    except subprocess.TimeoutExpired:
        print(f'  yt-dlp timeout for {genre}.')
    files = glob.glob(f'{out_dir}/*.wav') + glob.glob(f'{out_dir}/*.mp3')
    print(f'  {len(files)} files downloaded for {genre}')
    return files


print(f'✅ Audio pipeline ready.')
print(f'   extract_audio_features: {AUDIO_FEAT_DIM}-dim')
print(f'   extract_mfcc_2d       : {MFCC_2D_DIM}-dim  (MFCC+Δ+Δ²×{TIME_FRAMES})')


## 🔢 Step 8: Multi-Algorithm Clustering Engine

Uses NB1's robust DBSCAN eps-sweep strategy + NB2's Cluster Purity + Agglomerative (Ward + Complete).

| Algorithm | Notes |
|-----------|-------|
| K-Means | Fast, spherical |
| Agglomerative-Ward | Euclidean linkage |
| Agglomerative-Complete | Works on cosine-normalized embeddings |
| DBSCAN | Auto-tuned eps, noise-aware |

**6 Metrics:** Silhouette · Davies-Bouldin · Calinski-Harabasz · ARI · NMI · **Purity**


In [ ]:
def _fmt(v):
    """Format metric float for console."""
    return f'{v:+.3f}' if (isinstance(v, float) and not np.isnan(v)) else '   NaN'


def cluster_purity(y_true_masked, cluster_labels_masked):
    """Compute cluster purity on already-masked arrays (noise excluded)."""
    if len(y_true_masked) == 0:
        return np.nan
    yt = LabelEncoder().fit_transform(y_true_masked)
    cl = cluster_labels_masked
    total = sum(np.bincount(yt[cl == k]).max() for k in np.unique(cl))
    return total / len(yt)


def compute_metrics(Z, y_true, cluster_labels):
    """
    Compute all 6 metrics on noise-free subset.
    Returns dict: sil, db, ch, nmi, ari, purity

    ── NB2 technique (exact) ────────────────────────────────────────────────
    1. Guard degenerate Z (NaN / Inf / collapsed std < 1e-6) BEFORE anything.
       Collapsed latent space → all pairwise distances = 0 → silhouette = 0/0 = NaN.
    2. Minimum-sample check: len(Zm) >= 10  (NB2 threshold, not n_cl+1).
    3. NMI uses average_method='arithmetic'  (NB2 exact call).
    """
    nan = np.nan   # use np.nan for pandas / np.isnan() compatibility

    # ── Guard: invalid / degenerate Z ────────────────────────────────────────
    if Z is None or len(Z) == 0:
        return dict(sil=nan, db=nan, ch=nan, nmi=nan, ari=nan, purity=nan)
    if np.any(np.isnan(Z)) or np.any(np.isinf(Z)):
        return dict(sil=nan, db=nan, ch=nan, nmi=nan, ari=nan, purity=nan)
    if np.std(Z) < 1e-6:   # collapsed latent space (same guard as NB2)
        return dict(sil=nan, db=nan, ch=nan, nmi=nan, ari=nan, purity=nan)

    mask = cluster_labels != -1
    Zm   = Z[mask]
    ym   = np.asarray(y_true)[mask]
    cm   = cluster_labels[mask]
    n_cl = len(set(cm))

    # ── Guard: too few samples or clusters (NB2 technique: < 10 samples) ────
    if n_cl < 2 or len(Zm) < 10:
        return dict(sil=nan, db=nan, ch=nan, nmi=nan, ari=nan, purity=nan)

    return dict(
        sil    = silhouette_score(Zm, cm),
        db     = davies_bouldin_score(Zm, cm),
        ch     = calinski_harabasz_score(Zm, cm),
        nmi    = normalized_mutual_info_score(ym, cm, average_method='arithmetic'),
        ari    = adjusted_rand_score(ym, cm),
        purity = cluster_purity(ym, cm),
    )


def _nan_metrics():
    """Return an all-NaN metrics dict (used when Z is degenerate)."""
    n = np.nan
    return dict(sil=n, db=n, ch=n, nmi=n, ari=n, purity=n)


def run_clustering(Z, y_true, n_class, tag=''):
    """
    Run KMeans, Agglomerative (Ward + Complete), DBSCAN.
    Returns dict: algo → {labels, metrics, [n_found, noise_pct, eps]}

    ── NB2 technique (exact) ────────────────────────────────────────────────
    Degenerate Z guard at the TOP of the function — same as NB2.
    If Z is NaN/Inf or the latent space is collapsed (std < 1e-6), every
    algorithm would produce meaningless clusters, so we return NaN for all
    metrics immediately rather than propagating 0/0 into silhouette_score.
    """
    K       = n_class
    results = {}

    # ── NB2 upfront guard: skip ALL clustering if Z is degenerate ────────────
    _degenerate = (
        Z is None or len(Z) == 0
        or np.any(np.isnan(Z)) or np.any(np.isinf(Z))
        or float(np.std(Z)) < 1e-6
    )
    if _degenerate:
        reason = ('NaN/Inf in Z' if (Z is not None and (np.any(np.isnan(Z)) or np.any(np.isinf(Z))))
                  else 'latent space collapsed (std<1e-6)')
        if tag:
            print(f'  [{tag}] SKIP clustering — {reason}')
        _dummy_labels = np.zeros(len(Z) if Z is not None else 0, dtype=int)
        _dummy_db = dict(labels=_dummy_labels, eps=0.0, noise_pct=100.0,
                         n_found=0, metrics=_nan_metrics())
        _dummy    = dict(labels=_dummy_labels, metrics=_nan_metrics())
        return dict(KMeans=_dummy, Agglomerative_Ward=_dummy,
                    Agglomerative_Complete=_dummy, DBSCAN=_dummy_db)

    # ── K-Means ───────────────────────────────────────────────────────────────
    km = KMeans(n_clusters=K, n_init=KMEANS_NINIT, random_state=42).fit(Z)
    results['KMeans'] = {'labels': km.labels_, 'metrics': compute_metrics(Z, y_true, km.labels_)}

    # ── Agglomerative Ward ────────────────────────────────────────────────────
    agg_w = AgglomerativeClustering(n_clusters=K, linkage='ward').fit(Z)
    results['Agglomerative_Ward'] = {
        'labels': agg_w.labels_,
        'metrics': compute_metrics(Z, y_true, agg_w.labels_),
    }

    # ── Agglomerative Complete ────────────────────────────────────────────────
    agg_c = AgglomerativeClustering(n_clusters=K, linkage='complete').fit(Z)
    results['Agglomerative_Complete'] = {
        'labels': agg_c.labels_,
        'metrics': compute_metrics(Z, y_true, agg_c.labels_),
    }

    # ── DBSCAN — eps auto-tuned via percentile sweep on L2-normalised Z ──────
    # (NB2 exact technique: sweep pct 5→95, pick closest to K clusters with noise<30%)
    Z_norm    = normalize(Z, norm='l2')
    min_samp  = max(3, len(Z) // (K * 10))
    nbrs      = NearestNeighbors(n_neighbors=min_samp).fit(Z_norm)
    dists, _  = nbrs.kneighbors(Z_norm)
    kth_dists = np.sort(dists[:, -1])

    best_labels, best_eps, best_n = None, None, -1
    for pct in range(5, 96, 5):
        eps_try   = float(np.percentile(kth_dists, pct))
        l_try     = DBSCAN(eps=eps_try, min_samples=min_samp).fit_predict(Z_norm)
        n_try     = len(set(l_try)) - (1 if -1 in l_try else 0)
        noise_try = (l_try == -1).mean()
        if n_try >= 2 and noise_try < 0.30:
            if best_n == -1 or abs(n_try - K) < abs(best_n - K):
                best_labels, best_eps, best_n = l_try, eps_try, n_try

    if best_labels is None:   # fallback
        best_eps    = float(np.percentile(kth_dists, 50))
        best_labels = DBSCAN(eps=best_eps, min_samples=min_samp).fit_predict(Z_norm)
        best_n      = len(set(best_labels)) - (1 if -1 in best_labels else 0)

    noise_pct = float((best_labels == -1).mean() * 100)
    mask_db   = best_labels != -1
    db_metrics = (
        compute_metrics(Z_norm[mask_db], y_true[mask_db], best_labels[mask_db])
        if best_n >= 2 and mask_db.sum() > 1
        else _nan_metrics()
    )

    results['DBSCAN'] = {
        'labels':    best_labels,
        'eps':       best_eps,
        'noise_pct': noise_pct,
        'n_found':   best_n,
        'metrics':   db_metrics,
    }

    if tag:
        print(f'  [{tag}]')
        for algo, r in results.items():
            m = r['metrics']
            extra = (f'  clusters={r["n_found"]}  noise={r["noise_pct"]:.0f}%'
                     if algo == 'DBSCAN' else '')
            print(f'    {algo:<25} '
                  f'Sil={_fmt(m["sil"])}  DB={_fmt(m["db"])}  '
                  f'NMI={_fmt(m["nmi"])}  ARI={_fmt(m["ari"])}  '
                  f'Pur={_fmt(m["purity"])}{extra}')
    return results


print('✅ Clustering engine ready — KMeans · Agglom-Ward · Agglom-Complete · DBSCAN')
print('   Metrics: Silhouette | DB | CH | NMI | ARI | Purity (6 total)')
print('   [NB2 technique] Degenerate-Z guard · len>=10 threshold · NMI arithmetic')


## 📈 Step 9: Elbow Analysis Helper

In [ ]:
def elbow_analysis(Z, k_range=range(2, 16)):
    """Inertia, silhouette, CH across k values. Returns optimal_k via silhouette argmax.

    ── NB2 technique: same degenerate-Z guards used in run_clustering / compute_metrics.
    """
    k_range = [k for k in k_range if k < len(Z)]
    if len(k_range) < 2:
        print('  elbow_analysis: too few samples.')
        return {}
    # ── NB2-style guards ──────────────────────────────────────────────────────
    if Z is None or len(Z) == 0:
        print('  elbow_analysis: empty Z.')
        return {}
    if np.any(np.isnan(Z)) or np.any(np.isinf(Z)):
        print('  elbow_analysis: Z contains NaN/Inf — skipping.')
        return {}
    if np.std(Z) < 1e-6:
        print('  elbow_analysis: latent space collapsed (std<1e-6) — skipping.')
        return {}
    inertias, sils, chs, dbs = [], [], [], []
    for k in k_range:
        km     = KMeans(n_clusters=k, n_init=20, random_state=42).fit(Z)
        n_uniq = len(set(km.labels_))
        inertias.append(km.inertia_)
        if n_uniq < 2:
            sils.append(float('nan')); chs.append(float('nan')); dbs.append(float('nan'))
        else:
            sils.append(silhouette_score(Z, km.labels_))
            chs.append(calinski_harabasz_score(Z, km.labels_))
            dbs.append(davies_bouldin_score(Z, km.labels_))
    sils_arr  = np.array(sils, dtype=float)
    valid_idx = np.where(~np.isnan(sils_arr))[0]
    optimal_k = k_range[int(valid_idx[np.argmax(sils_arr[valid_idx])])] if len(valid_idx) > 0 else k_range[0]
    return dict(k_range=list(k_range), inertias=inertias,
                sil_scores=sils, ch_scores=chs, db_scores=dbs, optimal_k=optimal_k)

print('✅ elbow_analysis() ready (NaN-safe + NB2 degenerate-Z guard)')


## 🏋️ Step 10: Unified Training Engine (AdamW + Cosine LR + Early Stopping)

In [ ]:
def train_model(X_sc, model, y_onehot=None,
                epochs=EPOCHS, lr=LR, beta=1.0,
                batch_size=BATCH_SIZE, model_type='vae',
                audio_dim=None, verbose=True):
    """
    Unified training loop for all model types.

    model_type : 'vae' | 'ae' | 'cvae' | 'multimodal' | 'hybrid_conv'
    audio_dim  : required when model_type='multimodal'

    INPUT CONTRACT per model_type:
      'vae' on MLPVAE/BetaVAE/ConvVAE  → X_sc (StandardScaler audio, 65-dim)
      'vae' on Conv2DVAE                → X_conv2d = normalize_for_conv2d(X_raw_2d) (7680-dim)
      'ae'                              → X_sc (65-dim)
      'cvae'                            → X_sc (65-dim) + y_onehot
      'multimodal'                      → X_multimodal = np.hstack([X_sc, X_lyric_l2]) (193-dim)
      'hybrid_conv'                     → X_hybrid_conv = np.hstack([X_conv2d, X_lyric_l2]) (7808-dim)
      'vae' on MLPVAE for Hybrid-MLP    → X_hybrid (L2 audio ‖ L2 lyrics) (193-dim)
    """
    model = model.to(DEVICE)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    X_t   = torch.FloatTensor(X_sc)
    N     = len(X_t)
    idx   = torch.randperm(N, generator=torch.Generator().manual_seed(SEED))
    split = int(0.9 * N)
    tr_idx, va_idx = idx[:split], idx[split:]

    def make_ds(indices):
        Xs = X_t[indices]
        if y_onehot is not None:
            return TensorDataset(Xs, torch.FloatTensor(y_onehot)[indices])
        return TensorDataset(Xs)

    tr_loader = DataLoader(make_ds(tr_idx), batch_size=batch_size, shuffle=True,  drop_last=False)
    va_loader = DataLoader(make_ds(va_idx), batch_size=batch_size, shuffle=False, drop_last=False)

    best_val   = float('inf')
    best_state = None
    patience   = 0
    history    = []

    def _forward(batch):
        bx = batch[0].to(DEVICE)
        bc = batch[1].to(DEVICE) if len(batch) > 1 else None
        if model_type == 'ae':
            recon, _ = model(bx)
            return F.mse_loss(recon, bx)
        elif model_type == 'cvae':
            recon, mu, lv, _ = model(bx, bc)
            loss, _, _ = vae_loss_fn(recon, bx, mu, lv, beta)
            return loss
        elif model_type == 'multimodal':
            bx_audio = bx[:, :audio_dim]
            bx_lyric = bx[:, audio_dim:]
            recon, mu, lv, _ = model(bx_audio, bx_lyric)
            loss, _, _ = vae_loss_fn(recon, bx_audio, mu, lv, beta)
            return loss
        elif model_type == 'hybrid_conv':
            split_pt = N_MFCC_ROWS * TIME_FRAMES
            bx_conv  = bx[:, :split_pt]
            bx_lyric = bx[:, split_pt:]
            recon, mu, lv, _ = model(bx_conv, bx_lyric)
            loss, _, _ = vae_loss_fn(recon, bx_conv, mu, lv, beta)
            return loss
        else:   # 'vae'
            recon, mu, lv, _ = model(bx)
            loss, _, _ = vae_loss_fn(recon, bx, mu, lv, beta)
            return loss

    for epoch in range(1, epochs + 1):
        model.train()
        tr_sum = 0.0
        for batch in tr_loader:
            opt.zero_grad()
            loss = _forward(batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_sum += loss.item()
        tr_avg = tr_sum / len(tr_loader)

        model.eval()
        va_sum = 0.0
        with torch.no_grad():
            for batch in va_loader:
                va_sum += _forward(batch).item()
        va_avg = va_sum / len(va_loader)

        current_lr = sched.get_last_lr()[0]
        sched.step()
        history.append((tr_avg, va_avg, current_lr))

        if va_avg < best_val:
            best_val   = va_avg
            best_state = copy.deepcopy(model.state_dict())
            patience   = 0
        else:
            patience += 1

        if verbose and (epoch % 25 == 0 or epoch == 1):
            print(f'    Ep {epoch:3d}/{epochs}  '
                  f'train={tr_avg:.4f}  val={va_avg:.4f}  '
                  f'lr={current_lr:.2e}  patience={patience}/{EARLY_STOP_PATIENCE}')

        if patience >= EARLY_STOP_PATIENCE:
            if verbose:
                print(f'    Early stop at epoch {epoch}  (best val={best_val:.4f})')
            break

    model.load_state_dict(best_state)
    return model, history, best_val


def extract_latent(model, X_sc, batch_size=BATCH_SIZE,
                   model_type='vae', audio_dim=None):
    """Extract latent mu vectors. model_type matches train_model contract."""
    model.eval()
    X_t    = torch.FloatTensor(X_sc)
    Z_list = []
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            batch = X_t[i:i+batch_size].to(DEVICE)
            if model_type == 'multimodal':
                mu = model.get_latent(batch[:, :audio_dim], batch[:, audio_dim:])
            elif model_type == 'hybrid_conv':
                split_pt = N_MFCC_ROWS * TIME_FRAMES
                mu = model.get_latent(batch[:, :split_pt], batch[:, split_pt:])
            else:
                mu = model.get_latent(batch)
            Z_list.append(mu.cpu().numpy())
    return np.vstack(Z_list)


print('✅ Training engine ready:')
print('   train_model()    — vae | ae | cvae | multimodal | hybrid_conv')
print('   extract_latent() — dispatches get_latent() per model type')
print('   Features: AdamW + CosineAnnealing + early stopping + grad clipping')


## 📥 Step 11: Download GTZAN Dataset

In [ ]:
GTZAN_AUDIO = os.path.join(GTZAN_DIR, 'genres')

def _stream_dl(url, dest, desc=''):
    r = requests.get(url, stream=True, timeout=600)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(dest, 'wb') as f, tqdm(
        total=total, unit='B', unit_scale=True, unit_divisor=1024, desc=desc, ncols=80
    ) as bar:
        for chunk in r.iter_content(65536):
            f.write(chunk); bar.update(len(chunk))

def _resolve_gtzan_audio(search_root):
    wavs = glob.glob(f'{search_root}/**/*.wav', recursive=True)
    if not wavs:
        return search_root, []
    audio_root = os.path.dirname(wavs[0]).rsplit(os.sep, 1)[0]
    return audio_root, wavs

existing_wav = glob.glob(f'{GTZAN_AUDIO}/**/*.wav', recursive=True)
if not existing_wav:
    existing_wav = glob.glob(f'{GTZAN_DIR}/**/*.wav', recursive=True)

if len(existing_wav) >= 900:
    print(f'✅ GTZAN cached: {len(existing_wav)} .wav files.')
    GTZAN_AUDIO, _ = _resolve_gtzan_audio(GTZAN_DIR)
else:
    GTZAN_TAR = os.path.join(GTZAN_DIR, 'genres.tar.gz')
    GTZAN_MIRRORS = [
        'http://opihi.cs.uvic.ca/sound/genres.tar.gz',
        'https://huggingface.co/datasets/marsyas/gtzan/resolve/main/data/genres.tar.gz?download=true',
        'https://archive.org/download/gtzan_genre/genres.tar.gz',
    ]
    downloaded_via_tar    = False
    downloaded_via_kaggle = False

    for url in GTZAN_MIRRORS:
        try:
            print(f'⬇️  Trying: {url}')
            _stream_dl(url, GTZAN_TAR, desc='GTZAN')
            if os.path.getsize(GTZAN_TAR) < 10_000_000:
                print('  File too small — likely error page, skipping.')
                os.remove(GTZAN_TAR); continue
            downloaded_via_tar = True; break
        except Exception as e:
            print(f'  Mirror failed: {e}')

    if not downloaded_via_tar:
        print('  All HTTP mirrors failed — trying Kaggle …')
        r2 = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d',
             'andradaolteanu/gtzan-dataset-music-genre-classification',
             '-p', GTZAN_DIR, '--unzip'],
            capture_output=True, text=True, timeout=600,
        )
        _, found = _resolve_gtzan_audio(GTZAN_DIR)
        if found:
            print(f'  GTZAN via Kaggle: {len(found)} files.')
            downloaded_via_kaggle = True
        else:
            raise RuntimeError(
                'All GTZAN sources failed.\n'
                'Option A: Upload genres.tar.gz to /content/gtzan/ manually.\n'
                'Option B: Place kaggle.json at ~/.kaggle/ and re-run.'
            )

    if downloaded_via_tar and os.path.exists(GTZAN_TAR):
        print('📦 Extracting GTZAN …')
        with tarfile.open(GTZAN_TAR, 'r:gz') as tf:
            for m in tqdm(tf.getmembers(), desc='Extracting', unit='file', ncols=80):
                tf.extract(m, GTZAN_DIR)

    GTZAN_AUDIO, existing_wav = _resolve_gtzan_audio(GTZAN_DIR)
    if len(existing_wav) < 900:
        print(f'  WARNING: only {len(existing_wav)} .wav files found.')
    else:
        print(f'✅ GTZAN ready: {len(existing_wav)} .wav files')

print(f'\nGTZAN_AUDIO = {GTZAN_AUDIO}')
print('BanglaGITI + BMGCD will be loaded in Step 12.')
print('Ensure ~/.kaggle/kaggle.json exists for Kaggle downloads.')


## 🔧 Step 12: Build All Dataset Arrays

Extracts **both** 65-dim flat features **and** 7680-dim delta-stacked 2D MFCC from real audio.
No synthetic data anywhere.


In [ ]:
FEAT_DIM = AUDIO_FEAT_DIM    # 65

# ── GTZAN ──────────────────────────────────────────────────────────────────────
print('Loading GTZAN ...')
gtzan_wav = sorted(glob.glob(f'{GTZAN_AUDIO}/**/*.wav', recursive=True))
if not gtzan_wav:
    gtzan_wav = sorted(glob.glob(f'{GTZAN_DIR}/**/*.wav', recursive=True))
print(f'  Found {len(gtzan_wav)} .wav files.')

gc_g = defaultdict(int)
X_gtzan, X_gtzan_2d, y_gtzan, lang_gtzan, paths_gtzan = [], [], [], [], []
for fpath in tqdm(gtzan_wav, desc='  librosa GTZAN', leave=False):
    genre = os.path.basename(os.path.dirname(fpath))
    if gc_g[genre] >= N_PER_GENRE:
        continue
    feat    = extract_audio_features(fpath)
    feat_2d = extract_mfcc_2d(fpath)
    if feat is not None and feat_2d is not None:
        X_gtzan.append(feat); X_gtzan_2d.append(feat_2d)
        y_gtzan.append(genre); lang_gtzan.append('English')
        paths_gtzan.append(fpath); gc_g[genre] += 1

if len(X_gtzan) == 0:
    raise RuntimeError('GTZAN: no features extracted. Check Step 11 download.')
X_gtzan    = np.array(X_gtzan,    dtype=np.float32)
X_gtzan_2d = np.array(X_gtzan_2d, dtype=np.float32)
y_gtzan    = np.array(y_gtzan)
lang_gtzan = np.array(lang_gtzan)
print(f'✅ GTZAN : {X_gtzan.shape} | 2D: {X_gtzan_2d.shape} | Genres: {dict(Counter(y_gtzan))}')


# ── BanglaGITI ────────────────────────────────────────────────────────────────
print('\nLoading BanglaGITI ...')
bg_dir = download_kaggle_dataset('priyanjanasarkar/banglagiti', BANGLAGITI_DIR)
X_bg, X_bg_2d, y_bg, lang_bg, paths_bg = [], [], [], [], []
if bg_dir:
    recs_bg = collect_audio_from_dir(bg_dir, min_per_genre=0)
    gc_bg   = defaultdict(int)
    for fpath, genre in tqdm(recs_bg, desc='  librosa BanglaGITI', leave=False):
        if gc_bg[genre] >= N_PER_GENRE: continue
        feat    = extract_audio_features(fpath)
        feat_2d = extract_mfcc_2d(fpath)
        if feat is not None and feat_2d is not None:
            X_bg.append(feat); X_bg_2d.append(feat_2d)
            y_bg.append(genre); lang_bg.append('Bangla')
            paths_bg.append(fpath); gc_bg[genre] += 1

if len(X_bg) < MIN_SAMPLES_FOR_METRICS:
    print('  BanglaGITI unavailable — yt-dlp fallback ...')
    gc_fb = defaultdict(int)
    for genre, query in BANGLA_QUERIES_YT.items():
        files = download_bangla_yt(genre, query, n=N_PER_GENRE)
        for fpath in tqdm(files, desc=f'  librosa {genre}', leave=False):
            if gc_fb[genre] >= N_PER_GENRE: continue
            feat    = extract_audio_features(fpath)
            feat_2d = extract_mfcc_2d(fpath)
            if feat is not None and feat_2d is not None:
                X_bg.append(feat); X_bg_2d.append(feat_2d)
                y_bg.append(genre); lang_bg.append('Bangla')
                paths_bg.append(fpath); gc_fb[genre] += 1

if len(X_bg) == 0:
    print('  WARNING: BanglaGITI has 0 usable samples.')
    X_bg    = np.zeros((0, FEAT_DIM),    dtype=np.float32)
    X_bg_2d = np.zeros((0, MFCC_2D_DIM), dtype=np.float32)
    y_bg    = np.array([]); lang_bg = np.array([])
else:
    X_bg    = np.array(X_bg,    dtype=np.float32)
    X_bg_2d = np.array(X_bg_2d, dtype=np.float32)
    y_bg    = np.array(y_bg); lang_bg = np.array(lang_bg)
    print(f'✅ BanglaGITI: {X_bg.shape} | Genres: {dict(Counter(y_bg))}')


# ── BMGCD ──────────────────────────────────────────────────────────────────────
print('\nLoading BMGCD ...')
bm_dir = download_kaggle_dataset(
    'mdimranhassan/bangla-music-genre-classification', BMGCD_DIR)
X_bm, X_bm_2d, y_bm, lang_bm, paths_bm = [], [], [], [], []
if bm_dir:
    recs_bm = collect_audio_from_dir(bm_dir, min_per_genre=10)
    gc_bm   = defaultdict(int)
    for fpath, genre in tqdm(recs_bm, desc='  librosa BMGCD', leave=False):
        if gc_bm[genre] >= N_PER_GENRE: continue
        feat    = extract_audio_features(fpath)
        feat_2d = extract_mfcc_2d(fpath)
        if feat is not None and feat_2d is not None:
            X_bm.append(feat); X_bm_2d.append(feat_2d)
            y_bm.append(genre); lang_bm.append('Bangla')
            paths_bm.append(fpath); gc_bm[genre] += 1

if len(X_bm) < MIN_SAMPLES_FOR_METRICS:
    print('  BMGCD unavailable — yt-dlp fallback ...')
    gc_fb2 = defaultdict(int)
    for genre, query in BMGCD_QUERIES_YT.items():
        files = download_bangla_yt(genre, query, n=N_PER_GENRE)
        for fpath in tqdm(files, desc=f'  librosa {genre}', leave=False):
            if gc_fb2[genre] >= N_PER_GENRE: continue
            feat    = extract_audio_features(fpath)
            feat_2d = extract_mfcc_2d(fpath)
            if feat is not None and feat_2d is not None:
                X_bm.append(feat); X_bm_2d.append(feat_2d)
                y_bm.append(genre); lang_bm.append('Bangla')
                paths_bm.append(fpath); gc_fb2[genre] += 1

if len(X_bm) == 0:
    print('  WARNING: BMGCD has 0 usable samples.')
    X_bm    = np.zeros((0, FEAT_DIM),    dtype=np.float32)
    X_bm_2d = np.zeros((0, MFCC_2D_DIM), dtype=np.float32)
    y_bm    = np.array([]); lang_bm = np.array([])
else:
    X_bm    = np.array(X_bm,    dtype=np.float32)
    X_bm_2d = np.array(X_bm_2d, dtype=np.float32)
    y_bm    = np.array(y_bm); lang_bm = np.array(lang_bm)
    print(f'✅ BMGCD: {X_bm.shape} | Genres: {dict(Counter(y_bm))}')


# ── Record dicts for make_multimodal() ────────────────────────────────────────
def make_records(paths, y_labels, lang_labels):
    return [{'file': p, 'genre': g, 'language': l}
            for p, g, l in zip(paths, y_labels, lang_labels)]

records_gtzan = make_records(paths_gtzan, y_gtzan, lang_gtzan)
records_bg    = make_records(paths_bg,    y_bg,    lang_bg)
records_bm    = make_records(paths_bm,    y_bm,    lang_bm)

# ── Summary ────────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('  DATASET SUMMARY')
print('=' * 60)
for name, X, y in [('GTZAN', X_gtzan, y_gtzan),
                   ('BanglaGITI', X_bg, y_bg),
                   ('BMGCD', X_bm, y_bm)]:
    if len(X) > 0:
        print(f'  {name:<12}: {X.shape}  Genres: {len(np.unique(y))}  → {dict(Counter(y))}')
    else:
        print(f'  {name:<12}: EMPTY — skipped')
print('=' * 60)


## 🚀 Step 13: Full Experiment Pipeline

Trains **9 VAE/AE models + PCA + Raw-Spectral** on every dataset.
Uses NB1's datasets (GTZAN · BanglaGITI · BMGCD) with NB2's extended model suite.

| # | Model | Type |
|---|-------|------|
| 1 | MLP-VAE | Standard ELBO |
| 2 | Conv2D-VAE | 2D delta-stacked MFCC |
| 3 | HybridConvVAE | Conv2D + Lyric end-to-end |
| 4 | Hybrid-MLP-VAE | Audio flat ‖ Lyrics → MLP |
| 5 | Beta-VAE | β-sweep, best β selected per dataset |
| 6 | CVAE | Genre-conditional |
| 7 | Conv1D-VAE | 1D conv encoder |
| 8 | Autoencoder | Deterministic baseline |
| 9 | MultiModalVAE | Joint audio+lyric encoder |
| — | PCA | Linear baseline |
| — | Raw-Spectral | Direct feature clustering |


In [ ]:
def full_pipeline(X_raw, y_labels, lang_labels, dataset_name,
                  file_paths=None, X_raw_2d=None, scaler=None):
    """
    Full pipeline: 9 models + PCA baseline + raw spectral clustering.

    X_raw    : (N, 65)   — 65-dim audio features
    X_raw_2d : (N, 7680) — delta-stacked MFCC for Conv2DVAE / HybridConvVAE
    scaler   : fitted StandardScaler (pass scaler_all for cross-dataset consistency)
    """
    if len(X_raw) == 0:
        print(f'  SKIP {dataset_name} — empty dataset.')
        return None

    SEP = '=' * 70
    print(f'\n{SEP}')
    print(f'  DATASET : {dataset_name}')
    print(f'  Samples={len(X_raw)} | Features={X_raw.shape[1]} | '
          f'Genres={len(np.unique(y_labels))}')
    print(SEP)

    # ── Scaling ───────────────────────────────────────────────────────────────
    if scaler is not None:
        X_sc = scaler.transform(X_raw).astype(np.float32)
    else:
        scaler = StandardScaler()
        X_sc   = scaler.fit_transform(X_raw).astype(np.float32)

    le      = LabelEncoder()
    y_true  = le.fit_transform(y_labels)
    n_class = len(le.classes_)
    pca_dim = min(LATENT_DIM, X_sc.shape[1], X_sc.shape[0] - 1)

    # ── Records for lyrics ────────────────────────────────────────────────────
    records = [
        {'file':     file_paths[i] if file_paths is not None else None,
         'genre':    str(y_labels[i]),
         'language': str(lang_labels[i])}
        for i in range(len(X_raw))
    ]

    # ── Hybrid features (audio + lyrics) ─────────────────────────────────────
    print('  Building multi-modal features (audio + real lyrics)...')
    X_hybrid, has_real, X_lyric_l2 = make_multimodal(X_raw, records)
    has_real    = np.array(has_real, dtype=bool)
    X_hybrid_sc = X_hybrid.astype(np.float32)   # already L2 normalized, no further scaling
    X_multimodal = np.hstack([X_sc, X_lyric_l2]).astype(np.float32)
    C_oh = make_genre_onehot(y_labels, le)

    # ── Conv2D prep ───────────────────────────────────────────────────────────
    if X_raw_2d is not None and len(X_raw_2d) > 0:
        X_conv2d    = normalize_for_conv2d(X_raw_2d)
        X_conv2d    = align_for_conv2d(X_conv2d)
        _has_conv2d = True
    else:
        X_conv2d    = None
        _has_conv2d = False
        print('  [Conv2DVAE] X_raw_2d not provided — skipping Conv2D branch')

    # ── [1] MLP-VAE ───────────────────────────────────────────────────────────
    print('  [1/9] MLP-VAE …')
    m_mlp, mlp_hist, mlp_loss = train_model(
        X_sc, MLPVAE(X_sc.shape[1], LATENT_DIM).to(DEVICE), model_type='vae', beta=1.0)
    Z_mlp = extract_latent(m_mlp, X_sc, model_type='vae')

    # ── [2] Conv2D-VAE ────────────────────────────────────────────────────────
    if _has_conv2d:
        print(f'  [2/9] Conv2DVAE ({N_MFCC_ROWS}×{TIME_FRAMES}) …')
        m_conv, conv_hist, conv_loss = train_model(
            X_conv2d, Conv2DVAE().to(DEVICE), model_type='vae', beta=1.0)
        Z_conv = extract_latent(m_conv, X_conv2d, model_type='vae')
    else:
        print('  [2/9] Conv2DVAE fallback → MLP-VAE on 65-dim …')
        m_conv, conv_hist, conv_loss = train_model(
            X_sc, MLPVAE(X_sc.shape[1], LATENT_DIM).to(DEVICE), model_type='vae', beta=1.0)
        Z_conv = extract_latent(m_conv, X_sc, model_type='vae')

    # ── [3] HybridConvVAE ─────────────────────────────────────────────────────
    if _has_conv2d:
        print('  [3/9] HybridConvVAE (end-to-end Conv+Lyric) …')
        X_hybrid_conv = np.hstack([X_conv2d, X_lyric_l2]).astype(np.float32)
        m_hyb_conv, hyb_conv_hist, hyb_conv_loss = train_model(
            X_hybrid_conv, HybridConvVAE().to(DEVICE),
            model_type='hybrid_conv', beta=1.0)
        Z_hyb_conv = extract_latent(m_hyb_conv, X_hybrid_conv, model_type='hybrid_conv')
    else:
        print('  [3/9] HybridConvVAE fallback → Hybrid-MLP-VAE …')
        m_hyb_conv, hyb_conv_hist, hyb_conv_loss = train_model(
            X_hybrid_sc, MLPVAE(X_hybrid_sc.shape[1], LATENT_DIM).to(DEVICE),
            model_type='vae', beta=1.0)
        Z_hyb_conv = extract_latent(m_hyb_conv, X_hybrid_sc, model_type='vae')

    # ── [4] Hybrid-MLP-VAE ────────────────────────────────────────────────────
    print('  [4/9] Hybrid-MLP-VAE …')
    m_hyb_mlp, hyb_mlp_hist, hyb_mlp_loss = train_model(
        X_hybrid_sc, MLPVAE(X_hybrid_sc.shape[1], LATENT_DIM).to(DEVICE),
        model_type='vae', beta=1.0)
    Z_hyb_mlp = extract_latent(m_hyb_mlp, X_hybrid_sc, model_type='vae')

    # ── [5] Beta-VAE sweep ────────────────────────────────────────────────────
    print(f'  [5/9] Beta-VAE sweep {BETA_VALUES} …')
    beta_sweep      = {}
    best_beta_sil   = -np.inf
    best_beta_val   = BETA_VAE_B
    best_beta_Z     = None; best_beta_model = None
    best_beta_hist  = None; best_beta_loss  = float('inf')
    for beta_val in BETA_VALUES:
        m_b, h_b, l_b = train_model(
            X_sc, BetaVAE(X_sc.shape[1], LATENT_DIM).to(DEVICE),
            model_type='vae', beta=beta_val, verbose=False)
        Z_b   = extract_latent(m_b, X_sc, model_type='vae')
        km_b  = KMeans(n_clusters=n_class, n_init=20, random_state=42).fit(Z_b)
        n_uniq = len(set(km_b.labels_))
        m_met  = compute_metrics(Z_b, y_true, km_b.labels_) if n_uniq >= 2 else                  dict(sil=float('nan'), db=float('nan'), ch=float('nan'),
                      nmi=float('nan'), ari=float('nan'), purity=float('nan'))
        beta_sweep[beta_val] = dict(metrics=m_met, Z=Z_b, model=m_b, hist=h_b)
        sil = m_met['sil']
        if not np.isnan(sil) and sil > best_beta_sil:
            best_beta_sil   = sil;   best_beta_val   = beta_val
            best_beta_Z     = Z_b;   best_beta_model = m_b
            best_beta_hist  = h_b;   best_beta_loss  = l_b
    print(f'    Best β={best_beta_val}  Sil={best_beta_sil:.4f}')
    m_beta    = best_beta_model; Z_beta    = best_beta_Z
    beta_hist = best_beta_hist;  beta_loss = best_beta_loss

    # ── [6] CVAE ──────────────────────────────────────────────────────────────
    print('  [6/9] CVAE …')
    m_cvae, cvae_hist, cvae_loss = train_model(
        X_sc, CVAE(X_sc.shape[1], n_class, LATENT_DIM).to(DEVICE),
        y_onehot=C_oh, model_type='cvae', beta=1.0)
    Z_cvae = extract_latent(m_cvae, X_sc, model_type='cvae')

    # ── [7] Conv1D-VAE ────────────────────────────────────────────────────────
    print('  [7/9] Conv1D-VAE …')
    m_conv1d, conv1d_hist, conv1d_loss = train_model(
        X_sc, ConvVAE(X_sc.shape[1], LATENT_DIM).to(DEVICE),
        model_type='vae', beta=1.0)
    Z_conv1d = extract_latent(m_conv1d, X_sc, model_type='vae')

    # ── [8] Autoencoder ───────────────────────────────────────────────────────
    print('  [8/9] Autoencoder …')
    m_ae, ae_hist, ae_loss = train_model(
        X_sc, Autoencoder(X_sc.shape[1], LATENT_DIM).to(DEVICE), model_type='ae')
    Z_ae = extract_latent(m_ae, X_sc, model_type='ae')

    # ── [9] MultiModalVAE ─────────────────────────────────────────────────────
    print('  [9/9] MultiModalVAE …')
    m_mm, mm_hist, mm_loss = train_model(
        X_multimodal,
        MultiModalVAE(AUDIO_FEAT_DIM, LYRIC_DIM, FUSION_DIM, LATENT_DIM).to(DEVICE),
        model_type='multimodal', audio_dim=AUDIO_FEAT_DIM, beta=1.0)
    Z_mm = extract_latent(m_mm, X_multimodal,
                          model_type='multimodal', audio_dim=AUDIO_FEAT_DIM)

    # ── PCA baseline ──────────────────────────────────────────────────────────
    print('  [PCA] PCA baseline …')
    Z_pca = PCA(n_components=pca_dim, random_state=42).fit_transform(X_sc)

    # ── Raw spectral ──────────────────────────────────────────────────────────
    print('  [Raw] Direct feature clustering …')
    cl_raw = run_clustering(X_sc, y_true, n_class, 'Raw-Spectral')

    # ── Elbow analysis ────────────────────────────────────────────────────────
    print('  Elbow analysis …')
    elbow = elbow_analysis(Z_mlp, k_range=range(2, min(22, n_class + 5)))

    # ── Clustering on all latent spaces ───────────────────────────────────────
    print('\n  --- Clustering ---')
    cl_mlp      = run_clustering(Z_mlp,      y_true, n_class, 'MLP-VAE')
    cl_conv     = run_clustering(Z_conv,     y_true, n_class, 'Conv2D-VAE')
    cl_hyb_conv = run_clustering(Z_hyb_conv, y_true, n_class, 'Hybrid-Conv-VAE')
    cl_hyb_mlp  = run_clustering(Z_hyb_mlp,  y_true, n_class, 'Hybrid-MLP-VAE')
    cl_beta     = run_clustering(Z_beta,     y_true, n_class, 'Beta-VAE')
    cl_cvae     = run_clustering(Z_cvae,     y_true, n_class, 'CVAE')
    cl_conv1d   = run_clustering(Z_conv1d,   y_true, n_class, 'Conv1D-VAE')
    cl_ae       = run_clustering(Z_ae,       y_true, n_class, 'Autoencoder')
    cl_mm       = run_clustering(Z_mm,       y_true, n_class, 'MultiModalVAE')
    cl_pca      = run_clustering(Z_pca,      y_true, n_class, 'PCA')

    return dict(
        name=dataset_name, X_sc=X_sc, y_true=y_true,
        y_labels=y_labels, lang_labels=lang_labels,
        le=le, n_class=n_class, elbow=elbow,
        has_real_lyrics=has_real, scaler=scaler,
        best_beta=best_beta_val, beta_sweep=beta_sweep,
        Z=dict(mlp=Z_mlp, conv=Z_conv,
               hyb_conv=Z_hyb_conv, hyb_mlp=Z_hyb_mlp,
               beta=Z_beta, cvae=Z_cvae,
               conv1d=Z_conv1d, ae=Z_ae,
               mm=Z_mm, pca=Z_pca, raw=X_sc),
        cl=dict(mlp=cl_mlp, conv=cl_conv,
                hyb_conv=cl_hyb_conv, hyb_mlp=cl_hyb_mlp,
                beta=cl_beta, cvae=cl_cvae,
                conv1d=cl_conv1d, ae=cl_ae,
                mm=cl_mm, pca=cl_pca, raw=cl_raw),
        hist=dict(mlp=mlp_hist, conv=conv_hist,
                  hyb_conv=hyb_conv_hist, hyb_mlp=hyb_mlp_hist,
                  beta=beta_hist, cvae=cvae_hist,
                  conv1d=conv1d_hist, ae=ae_hist, mm=mm_hist),
        loss=dict(mlp=mlp_loss, conv=conv_loss,
                  hyb_conv=hyb_conv_loss, hyb_mlp=hyb_mlp_loss,
                  beta=beta_loss, cvae=cvae_loss,
                  conv1d=conv1d_loss, ae=ae_loss, mm=mm_loss),
        models=dict(mlp=m_mlp, conv=m_conv,
                    hyb_conv=m_hyb_conv, hyb_mlp=m_hyb_mlp,
                    beta=m_beta, cvae=m_cvae,
                    conv1d=m_conv1d, ae=m_ae, mm=m_mm),
    )


# ── Fit a shared scaler on all available data ─────────────────────────────────
_parts = [X for X in [X_gtzan, X_bg, X_bm] if len(X) > 0]
X_all_for_scaler = np.vstack(_parts).astype(np.float32)
scaler_all  = StandardScaler().fit(X_all_for_scaler)
print(f'scaler_all fitted on {X_all_for_scaler.shape[0]} samples (all datasets combined)')

# ── Run on all three datasets ─────────────────────────────────────────────────
all_results = {}

all_results['GTZAN'] = full_pipeline(
    X_gtzan, y_gtzan, lang_gtzan, 'GTZAN (English Multi-Genre)',
    file_paths=paths_gtzan, X_raw_2d=X_gtzan_2d, scaler=scaler_all)

if len(X_bg) > 0:
    all_results['BanglaGITI'] = full_pipeline(
        X_bg, y_bg, lang_bg, 'BanglaGITI (Bengali Songs)',
        file_paths=paths_bg, X_raw_2d=X_bg_2d, scaler=scaler_all)

if len(X_bm) > 0:
    all_results['BMGCD'] = full_pipeline(
        X_bm, y_bm, lang_bm, 'BMGCD (Bangla Music Genre)',
        file_paths=paths_bm, X_raw_2d=X_bm_2d, scaler=scaler_all)

print('\n✅ All experiments complete!')


## 🌀 Step 31: Diffusion Latent Refiner

**Core idea:** After any VAE is trained, its latent space `Z` may still have noisy, overlapping clusters. 
We train a tiny **DDPM-style denoising network directly on `Z`** (never on raw audio). 
The network learns the score function ∇ log p(z) — i.e. which direction pushes each point toward higher density. 
At inference we run **DDIM deterministic reverse steps** starting from a lightly-noised `Z`, 
producing `Z_refined` where points migrate toward cluster modes.

- **Fully unsupervised** — no genre labels used anywhere
- **Post-hoc** — applied after training; no VAE retraining needed
- **Architecture:** MLP score-net with sinusoidal time embedding | linear β-schedule
- **Sampling:** DDIM deterministic (η=0) for stable, reproducible refinement

> *References: Ho et al. DDPM (2020) · Song et al. DDIM (2021) · DiffAE latent diffusion (2022)*

In [ ]:
import math, copy

# ══════════════════════════════════════════════════════════════════════════════
# 1.  SINUSOIDAL TIME EMBEDDING
# ══════════════════════════════════════════════════════════════════════════════
class SinusoidalTimeEmb(nn.Module):
    """Classic sinusoidal positional encoding adapted for diffusion timesteps."""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t):          # t : (B,) long
        half  = self.dim // 2
        freqs = torch.exp(
            -math.log(10_000) *
            torch.arange(half, device=t.device, dtype=torch.float32) / half
        )
        args  = t[:, None].float() * freqs[None]   # (B, half)
        return torch.cat([args.sin(), args.cos()], dim=-1)  # (B, dim)


# ══════════════════════════════════════════════════════════════════════════════
# 2.  SCORE NETWORK  — predicts noise ε added at timestep t
# ══════════════════════════════════════════════════════════════════════════════
class LatentScoreNet(nn.Module):
    """
    Input  : z_t (B × z_dim) + t (B,) → predicted_noise (B × z_dim)
    Design : 3-layer MLP with SiLU + LayerNorm for training stability.
    """
    def __init__(self, z_dim: int = LATENT_DIM, t_dim: int = 64, h_dim: int = 256):
        super().__init__()
        self.time_emb = SinusoidalTimeEmb(t_dim)
        self.net = nn.Sequential(
            nn.Linear(z_dim + t_dim, h_dim),
            nn.SiLU(),
            nn.LayerNorm(h_dim),
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.LayerNorm(h_dim),
            nn.Linear(h_dim, h_dim // 2),
            nn.SiLU(),
            nn.Linear(h_dim // 2, z_dim),
        )

    def forward(self, z_t, t):
        te  = self.time_emb(t)                    # (B, t_dim)
        inp = torch.cat([z_t, te], dim=1)         # (B, z_dim+t_dim)
        return self.net(inp)                      # (B, z_dim)


# ══════════════════════════════════════════════════════════════════════════════
# 3.  LINEAR NOISE SCHEDULE
# ══════════════════════════════════════════════════════════════════════════════
class LinearNoiseSchedule:
    """Pre-computes √ᾱ_t and √(1−ᾱ_t) for forward/reverse diffusion."""
    def __init__(self, T: int = 200, beta_start: float = 1e-4, beta_end: float = 0.02):
        self.T        = T
        betas         = torch.linspace(beta_start, beta_end, T)
        alpha_bar     = torch.cumprod(1.0 - betas, dim=0)
        self.betas    = betas
        self.alpha_bar  = alpha_bar
        self.sqrt_ab    = alpha_bar.sqrt()
        self.sqrt_1mab  = (1.0 - alpha_bar).sqrt()

    def to(self, device):
        for attr in ('betas', 'alpha_bar', 'sqrt_ab', 'sqrt_1mab'):
            setattr(self, attr, getattr(self, attr).to(device))
        return self

    def q_sample(self, z0, t, noise=None):
        """Forward: z_t = √ᾱ_t · z0 + √(1−ᾱ_t) · ε"""
        if noise is None:
            noise = torch.randn_like(z0)
        s_ab  = self.sqrt_ab[t].unsqueeze(-1)     # (B,1)
        s_1ab = self.sqrt_1mab[t].unsqueeze(-1)
        return s_ab * z0 + s_1ab * noise, noise


# ══════════════════════════════════════════════════════════════════════════════
# 4.  TRAINING FUNCTION
# ══════════════════════════════════════════════════════════════════════════════
def train_diffusion_refiner(
        Z_latent: np.ndarray,
        z_dim:      int   = LATENT_DIM,
        T_steps:    int   = 200,
        diff_epochs:int   = 150,
        diff_lr:    float = 2e-4,
        batch_size: int   = 256,
        patience:   int   = 20,
        verbose:    bool  = True,
):
    """
    Train a DDPM denoising score-network on a pre-computed latent array.
    Returns (score_net, schedule, Z_mean, Z_std) — everything needed to refine.
    """
    schedule  = LinearNoiseSchedule(T=T_steps).to(DEVICE)
    score_net = LatentScoreNet(z_dim=z_dim).to(DEVICE)
    opt       = torch.optim.AdamW(score_net.parameters(), lr=diff_lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=diff_epochs)

    # Standardise latent (critical for diffusion stability)
    Z_t    = torch.FloatTensor(Z_latent)
    Z_mean = Z_t.mean(0)
    Z_std  = Z_t.std(0).clamp(min=1e-8)
    Z_norm = (Z_t - Z_mean) / Z_std

    loader = DataLoader(
        TensorDataset(Z_norm),
        batch_size=min(batch_size, len(Z_norm)),
        shuffle=True, drop_last=False,
    )

    best_loss, best_state, pat_count = float('inf'), None, 0

    for epoch in range(1, diff_epochs + 1):
        score_net.train()
        ep_loss = 0.0
        for (z0,) in loader:
            z0    = z0.to(DEVICE)
            t     = torch.randint(0, T_steps, (z0.size(0),), device=DEVICE)
            z_noisy, noise = schedule.q_sample(z0, t)
            loss  = F.mse_loss(score_net(z_noisy, t), noise)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(score_net.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item()
        ep_loss /= max(len(loader), 1)
        scheduler.step()

        if ep_loss < best_loss:
            best_loss, best_state, pat_count = ep_loss, copy.deepcopy(score_net.state_dict()), 0
        else:
            pat_count += 1

        if verbose and (epoch % 30 == 0 or epoch == 1):
            print(f'    [DiffRefiner] Ep {epoch:3d}  loss={ep_loss:.6f}  pat={pat_count}')
        if pat_count >= patience:
            if verbose:
                print(f'    [DiffRefiner] Early stop at epoch {epoch}')
            break

    score_net.load_state_dict(best_state)
    return score_net, schedule, Z_mean, Z_std


# ══════════════════════════════════════════════════════════════════════════════
# 5.  DDIM REFINEMENT INFERENCE
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def diffusion_refine_latent(
        score_net,
        schedule,
        Z_latent:      np.ndarray,
        Z_mean,
        Z_std,
        noise_level:   float = 0.30,   # how aggressively to perturb before refining
        refine_steps:  int   = 50,     # number of DDIM reverse steps
        batch_size:    int   = 512,
) -> np.ndarray:
    """
    DDIM deterministic reverse diffusion to refine latent representations.

    Strategy:
      1. Standardise Z → Z_norm
      2. Add light noise (t = T//3) to preserve structure
      3. Run DDIM reverse steps t → 0  (η=0, fully deterministic)
      4. Denormalise back to original latent scale

    noise_level=0.30 means we start at ~30% noise — enough to smooth intra-
    cluster scatter without destroying inter-cluster geometry.
    """
    score_net.eval()
    T = schedule.T

    Z_norm = ((torch.FloatTensor(Z_latent) - Z_mean) / Z_std).to(DEVICE)

    # Start: lightly-noised version of Z_norm  (t_start = T//3)
    t_start  = T // 3
    z_noisy  = (
        schedule.sqrt_ab[t_start]   * Z_norm +
        schedule.sqrt_1mab[t_start] * torch.randn_like(Z_norm) * noise_level
    )

    # Evenly-spaced DDIM timesteps from t_start → 1
    timesteps = torch.linspace(t_start, 1, refine_steps + 1).long()

    Z_out = []
    for i in range(0, len(Z_norm), batch_size):
        zb = z_noisy[i : i + batch_size].clone()

        for step_i in range(len(timesteps) - 1):
            t_now  = int(timesteps[step_i])
            t_next = int(timesteps[step_i + 1])
            if t_now <= 0:
                break

            t_batch  = torch.full((zb.size(0),), t_now, device=DEVICE, dtype=torch.long)
            eps_pred = score_net(zb, t_batch)

            ab_now  = schedule.alpha_bar[t_now]
            ab_next = schedule.alpha_bar[max(t_next - 1, 0)]

            # DDIM update  (η=0 — no stochastic noise added)
            x0_pred = (zb - (1 - ab_now).sqrt() * eps_pred) / ab_now.sqrt().clamp(min=1e-8)
            x0_pred = x0_pred.clamp(-5.0, 5.0)
            zb      = ab_next.sqrt() * x0_pred + (1 - ab_next).sqrt() * eps_pred

        Z_out.append(zb.cpu())

    Z_refined_norm = torch.cat(Z_out, dim=0).numpy()
    # Denormalise back to original latent scale
    return (Z_refined_norm * Z_std.numpy() + Z_mean.numpy()).astype(np.float32)


print('✅ Step 31: Diffusion Latent Refiner defined')
print('   Classes   : SinusoidalTimeEmb | LatentScoreNet | LinearNoiseSchedule')
print('   Functions : train_diffusion_refiner()  |  diffusion_refine_latent()')
print('   Usage     : Call Steps 32-33 below after all_results is populated.')


## 🏃 Step 32: Apply Diffusion Refiner to All Datasets

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Step 32: Apply Diffusion Refiner to every dataset
#
# For each dataset we:
#   1. Pick the best-performing existing VAE latent (by KMeans Silhouette)
#   2. Train the score-net on that latent
#   3. Run DDIM refinement to get Z_refined
#   4. Re-cluster Z_refined and compare metrics
# ══════════════════════════════════════════════════════════════════════════════

diff_results = {}   # ds_key → {Z_base_key, Z_refined, cl_refined, metrics_delta}

VAE_CANDIDATE_KEYS = ['mlp', 'conv', 'hyb_conv', 'hyb_mlp',
                      'beta', 'cvae', 'conv1d', 'ae', 'mm']

SEP = '=' * 72
for ds_key, res in all_results.items():
    if res is None:
        continue

    print(f'\n{SEP}')
    print(f'  Diffusion Latent Refiner — {ds_key}')
    print(f'  N={len(res["X_sc"])}  latent_dim={LATENT_DIM}  genres={res["n_class"]}')
    print(SEP)

    y_true  = res['y_true']
    n_class = res['n_class']

    # ── 1. Pick best existing VAE latent ────────────────────────────────────
    best_key, best_sil = 'mlp', -np.inf
    for zk in VAE_CANDIDATE_KEYS:
        if zk not in res['cl']:
            continue
        s = res['cl'][zk]['KMeans']['metrics'].get('sil', np.nan)
        if not np.isnan(s) and s > best_sil:
            best_sil, best_key = s, zk

    Z_base = res['Z'][best_key]
    base_label = MODEL_LABELS.get(best_key, best_key)
    print(f'  Base latent: {base_label}  (KMeans Sil = {best_sil:.4f})')

    # ── 2. Train denoiser on Z_base ─────────────────────────────────────────
    print('  Training score-net ...')
    score_net, schedule, Z_mean, Z_std = train_diffusion_refiner(
        Z_base,
        z_dim       = Z_base.shape[1],
        T_steps     = 200,
        diff_epochs = 150,
        diff_lr     = 2e-4,
        batch_size  = 256,
        patience    = 20,
        verbose     = True,
    )

    # ── 3. DDIM refinement ───────────────────────────────────────────────────
    print('  Running DDIM refinement (50 steps) ...')
    Z_refined = diffusion_refine_latent(
        score_net, schedule, Z_base, Z_mean, Z_std,
        noise_level  = 0.30,
        refine_steps = 50,
        batch_size   = 512,
    )

    # ── 4. Cluster Z_refined ────────────────────────────────────────────────
    print('  Clustering refined latent ...')
    cl_refined = run_clustering(Z_refined, y_true, n_class,
                                tag=f'DiffRefined({base_label})')

    # ── 5. Δ metrics vs base ────────────────────────────────────────────────
    base_cl = res['cl'][best_key]
    print(f'\n  ┌─ Metrics Comparison (KMeans) ───────────────────────────────')
    print(f'  │  Metric          Base ({base_label:<16})  Refined      Δ')
    print(f'  ├─────────────────────────────────────────────────────────────')
    for metric_key, metric_name in [('sil','Silhouette'), ('db','Davies-Bouldin'),
                                     ('ch','Calinski-H'), ('nmi','NMI'),
                                     ('ari','ARI'), ('purity','Purity')]:
        b = base_cl['KMeans']['metrics'].get(metric_key, np.nan)
        r = cl_refined['KMeans']['metrics'].get(metric_key, np.nan)
        if np.isnan(b) or np.isnan(r):
            continue
        delta = r - b
        # For Davies-Bouldin lower is better, flip sign for arrow
        better = delta > 0.001 if metric_key != 'db' else delta < -0.001
        arrow  = '🚀 ↑' if better else ('↓' if delta < -0.001 else '~')
        print(f'  │  {metric_name:<18} {b:>8.4f}           {r:>8.4f}   {delta:+.4f}  {arrow}')
    print(f'  └─────────────────────────────────────────────────────────────')

    diff_results[ds_key] = dict(
        base_key   = best_key,
        base_label = base_label,
        base_sil   = best_sil,
        Z_base     = Z_base,
        Z_refined  = Z_refined,
        cl_base    = base_cl,
        cl_refined = cl_refined,
        score_net  = score_net,
    )

print(f'\n{SEP}')
print('✅ Diffusion refinement complete for all datasets')
print(SEP)


## 📊 Step 33: Visualise Before vs After Refinement

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Step 33: Visualise — Before vs After Diffusion Refinement
#
# For each dataset: side-by-side UMAP (base vs refined) coloured by genre
# + bar chart of all 6 clustering metrics.
# ══════════════════════════════════════════════════════════════════════════════

for ds_key, dr in diff_results.items():
    res       = all_results[ds_key]
    n_class   = res['n_class']
    y_true    = res['y_true']
    PAL       = plt.colormaps['tab20'].resampled(n_class)
    n_nb      = min(30, len(dr['Z_base']) - 1)

    # ── UMAP projections ────────────────────────────────────────────────────
    reducer = umap.UMAP(n_components=2, n_neighbors=n_nb,
                        min_dist=0.1, random_state=42)
    Z2_base = reducer.fit_transform(dr['Z_base'])
    Z2_ref  = reducer.fit_transform(dr['Z_refined'])

    fig, axes = plt.subplots(2, 3, figsize=(21, 12))
    fig.suptitle(
        f'Diffusion Latent Refiner — {res["name"]}\n'
        f'Base: {dr["base_label"]} → Refined by DDPM/DDIM score-net',
        fontsize=13, fontweight='bold'
    )

    def scatter_genres(ax, Z2, title_suffix):
        for gi in range(n_class):
            m = y_true == gi
            if m.any():
                ax.scatter(Z2[m, 0], Z2[m, 1], c=[PAL(gi)], s=12,
                           alpha=0.7, linewidths=0,
                           label=res['le'].classes_[gi])
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(alpha=0.15)
        ax.set_title(title_suffix, fontsize=10, fontweight='bold')

    # Row 0 — Base
    km_base = dr['cl_base']['KMeans']
    sil_b   = km_base['metrics']['sil']
    nmi_b   = km_base['metrics']['nmi']
    scatter_genres(axes[0, 0], Z2_base,
                   f'BASE — {dr["base_label"]}\nGenres | Sil={sil_b:.4f}  NMI={nmi_b:.4f}')
    axes[0, 0].legend(fontsize=7, ncol=2, loc='upper right',
                      bbox_to_anchor=(1.0, 1.0), framealpha=0.6)

    axes[0, 1].scatter(Z2_base[:, 0], Z2_base[:, 1],
                       c=km_base['labels'], cmap='tab10', s=12, alpha=0.7, linewidths=0)
    axes[0, 1].set_title(f'BASE — K-Means clusters', fontsize=10, fontweight='bold')
    axes[0, 1].set_xticks([]); axes[0, 1].set_yticks([]); axes[0, 1].grid(alpha=0.15)

    # Row 1 — Refined
    km_ref  = dr['cl_refined']['KMeans']
    sil_r   = km_ref['metrics']['sil']
    nmi_r   = km_ref['metrics']['nmi']
    scatter_genres(axes[1, 0], Z2_ref,
                   f'REFINED — {dr["base_label"]} + Diffusion\nGenres | Sil={sil_r:.4f}  NMI={nmi_r:.4f}')

    axes[1, 1].scatter(Z2_ref[:, 0], Z2_ref[:, 1],
                       c=km_ref['labels'], cmap='tab10', s=12, alpha=0.7, linewidths=0)
    axes[1, 1].set_title(f'REFINED — K-Means clusters', fontsize=10, fontweight='bold')
    axes[1, 1].set_xticks([]); axes[1, 1].set_yticks([]); axes[1, 1].grid(alpha=0.15)

    # Column 2 — 6-metric bar comparison
    metric_names = ['Silhouette↑', 'Davies-Bouldin↓', 'Calinski-H↑', 'NMI↑', 'ARI↑', 'Purity↑']
    metric_keys  = ['sil', 'db', 'ch', 'nmi', 'ari', 'purity']
    # Normalise to [0,1] for bar display (DB flipped)
    vals_b, vals_r = [], []
    for mk in metric_keys:
        vb = km_base['metrics'].get(mk, 0) or 0
        vr = km_ref['metrics'].get(mk, 0) or 0
        vals_b.append(vb); vals_r.append(vr)

    # Merge rows 0,1 col 2 into one taller axis
    for ax_del in (axes[0, 2], axes[1, 2]):
        ax_del.set_visible(False)
    ax_bar = fig.add_subplot(2, 3, (3, 6))   # spans both rows, col 3

    x  = np.arange(len(metric_names))
    w  = 0.35
    b1 = ax_bar.bar(x - w/2, vals_b, w, label=f'Base ({dr["base_label"]})',
                    color='#1565C0', alpha=0.80)
    b2 = ax_bar.bar(x + w/2, vals_r, w, label='Diffusion Refined',
                    color='#D84315', alpha=0.80)
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        if abs(h) > 0.001:
            ax_bar.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=7)
    ax_bar.set_xticks(x); ax_bar.set_xticklabels(metric_names, rotation=30, ha='right', fontsize=9)
    ax_bar.set_title('All 6 Metrics — Base vs Refined (KMeans)', fontweight='bold', fontsize=10)
    ax_bar.legend(fontsize=9); ax_bar.grid(axis='y', alpha=0.3)
    ax_bar.axhline(0, color='black', lw=0.8)

    plt.tight_layout()
    fname = f'{OUTPUT_DIR}/diffusion_refiner_{ds_key.lower()}.png'
    plt.savefig(fname, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {fname}')


# ── Cross-dataset summary table ──────────────────────────────────────────────
print('\nCROSS-DATASET SUMMARY — Diffusion Refiner (KMeans)')
print('=' * 90)
print(f'{"Dataset":<14} {"Base Model":<20} {"Metric":<18} {"Base":>8} {"Refined":>8} {"Δ":>8}  Result')
print('-' * 90)
for ds_key, dr in diff_results.items():
    for mk, mn in [("sil","Silhouette"), ("nmi","NMI"), ("ari","ARI"), ("purity","Purity")]:
        b = dr['cl_base']['KMeans']['metrics'].get(mk, np.nan)
        r = dr['cl_refined']['KMeans']['metrics'].get(mk, np.nan)
        if np.isnan(b) or np.isnan(r): continue
        d = r - b
        tag = '🚀 IMPROVED' if d > 0.005 else ('❌ degraded' if d < -0.005 else '  ~ same')
        print(f'{ds_key:<14} {dr["base_label"]:<20} {mn:<18} {b:>8.4f} {r:>8.4f} {d:>+8.4f}  {tag}')
    print()


## 📊 Step 14: Genre Distribution Overview

In [ ]:
valid = [(k, v) for k, v in all_results.items() if v is not None]
fig, axes = plt.subplots(1, len(valid), figsize=(8 * len(valid), 5))
if len(valid) == 1:
    axes = [axes]
fig.suptitle('Genre Distribution per Dataset', fontsize=14, fontweight='bold')
for ax, (key, res) in zip(axes, valid):
    unique, counts = np.unique(res['y_labels'], return_counts=True)
    idx = np.argsort(counts)[::-1]
    bars = ax.bar(range(len(unique)), counts[idx],
                  color=plt.cm.tab20(np.linspace(0, 1, len(unique))))
    ax.set_xticks(range(len(unique)))
    ax.set_xticklabels(unique[idx], rotation=40, ha='right', fontsize=8)
    ax.set_title(res['name'], fontweight='bold'); ax.set_ylabel('Count'); ax.grid(axis='y', alpha=0.3)
    for bar, c in zip(bars, counts[idx]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(c), ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/genre_distribution.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved: genre_distribution.png')


## 🗺️ Step 15: t-SNE + UMAP Dimensionality Reduction

In [ ]:
SKIP_VIS = {'pca'}   # PCA already low-dim — use first 2 components directly

print('🔄 Computing t-SNE + UMAP (~5-10 min)...')
for key, res in all_results.items():
    if res is None: continue
    print(f'  Dataset: {key}'); res['vis'] = {}
    for zkey, Z in res['Z'].items():
        if zkey in SKIP_VIS:
            res['vis'][zkey] = {'tsne': Z[:, :2], 'umap': Z[:, :2]}
            print(f'    {zkey}... passthrough (2D already)')
            continue
        print(f'    {zkey}...', end=' ', flush=True)
        perp = min(40, max(5, Z.shape[0] // 3))
        n_nb = min(30, Z.shape[0] - 1)
        res['vis'][zkey] = {
            'tsne': TSNE(n_components=2, perplexity=perp, n_iter=1000,
                         random_state=42).fit_transform(Z),
            'umap': umap.UMAP(n_components=2, n_neighbors=n_nb,
                              min_dist=0.1, random_state=42).fit_transform(Z),
        }
        print(f'done (N={Z.shape[0]}, perp={perp}, n_nb={n_nb})')
print('✅ All reductions done.')


## 🎨 Step 16: Latent Space Plots — All Models (UMAP, Genre + Language)

In [ ]:
for key, res in all_results.items():
    if res is None: continue
    n_class  = res['n_class']
    PAL      = plt.colormaps['tab20'].resampled(n_class)
    n_models = len(Z_KEYS_ALL)
    fig, axes = plt.subplots(n_models, 2, figsize=(18, n_models * 3.5), squeeze=False)
    fig.suptitle(f'Latent Space (UMAP) — All Models — {res["name"]}',
                 fontsize=14, fontweight='bold')
    for row, zkey in enumerate(Z_KEYS_ALL):
        if zkey not in res['vis']: continue
        Z2 = res['vis'][zkey]['umap']
        cl = res['cl'][zkey]
        ml = MODEL_LABELS[zkey]
        # Left: genre colour
        ax = axes[row, 0]
        for gi in range(n_class):
            m = res['y_true'] == gi
            if m.any():
                ax.scatter(Z2[m, 0], Z2[m, 1], c=[PAL(gi)], s=8, alpha=0.6, linewidths=0)
        sil = cl['KMeans']['metrics']['sil']
        nmi = cl['KMeans']['metrics']['nmi']
        ax.set_title(f'{ml} | Genre | Sil={sil:.3f}  NMI={nmi:.3f}' if not np.isnan(sil) else f'{ml} | Genre',
                     fontsize=9, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(alpha=0.15)
        # Right: language separation
        ax = axes[row, 1]
        for lang, color, mk in [('English', '#1565C0', 'o'), ('Bangla', '#C62828', '^')]:
            lm = res['lang_labels'] == lang
            if lm.any():
                ax.scatter(Z2[lm, 0], Z2[lm, 1], c=color, marker=mk,
                           s=8, alpha=0.55, label=lang, linewidths=0)
        ax.set_title(f'{ml} | Language', fontsize=9, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(alpha=0.15)
        if row == 0: ax.legend(fontsize=8)
    plt.tight_layout()
    fname = f'{OUTPUT_DIR}/latent_all_{key.lower()}.png'
    plt.savefig(fname, dpi=110, bbox_inches='tight')
    plt.show(); print(f'✅ Saved: {fname}')


## 📈 Step 17: Elbow Method — Optimal K Selection

In [ ]:
valid_elbow = [(k, v) for k, v in all_results.items() if v is not None and v.get('elbow')]
if not valid_elbow:
    print('No valid elbow results.')
else:
    fig, axes = plt.subplots(len(valid_elbow), 3,
                             figsize=(18, len(valid_elbow) * 5), squeeze=False)
    fig.suptitle('Elbow Method  (red = true genres | green = suggested K)',
                 fontsize=14, fontweight='bold')
    for row, (key, res) in enumerate(valid_elbow):
        elbow = res['elbow']
        ks    = elbow['k_range']
        opt_k = elbow.get('optimal_k')
        for col, (vals, ylabel, title) in enumerate([
            (elbow['inertias'],   'Inertia',          'Inertia (↓)'),
            (elbow['sil_scores'], 'Silhouette Score',  'Silhouette (↑)'),
            (elbow['ch_scores'],  'Calinski-Harabasz', 'CH Index (↑)'),
        ]):
            ax = axes[row, col]
            ax.plot(ks, vals, 'o-', color='#1565C0', linewidth=2, markersize=5)
            ax.axvline(res['n_class'], color='red', linestyle='--', alpha=0.7,
                       label=f'K={res["n_class"]} (true)')
            if opt_k is not None:
                ax.axvline(opt_k, color='green', linestyle=':', alpha=0.7,
                           label=f'K={opt_k} (suggested)')
            ax.set_xlabel('K'); ax.set_ylabel(ylabel)
            ax.set_title(f'{key} — {title}', fontweight='bold')
            ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/elbow_plots.png', dpi=130, bbox_inches='tight')
    plt.show(); print('✅ Saved: elbow_plots.png')


## 🔍 Step 18: DBSCAN Cluster Analysis

In [ ]:
valid = [(k, v) for k, v in all_results.items() if v is not None]
fig, axes = plt.subplots(len(valid), len(Z_KEYS_ALL),
                         figsize=(7 * len(Z_KEYS_ALL), 5 * len(valid)), squeeze=False)
fig.suptitle('DBSCAN Results (grey = noise points)', fontsize=14, fontweight='bold')
for row, (key, res) in enumerate(valid):
    for col, zkey in enumerate(Z_KEYS_ALL):
        ax   = axes[row, col]
        Z2   = res['vis'][zkey]['umap']
        db   = res['cl'][zkey]['DBSCAN']
        lbls = db['labels']
        noise = lbls == -1
        ax.scatter(Z2[noise, 0], Z2[noise, 1], c='lightgrey', s=5, alpha=0.4)
        if (~noise).any():
            ax.scatter(Z2[~noise, 0], Z2[~noise, 1],
                       c=lbls[~noise], cmap='tab10', s=8, alpha=0.7, linewidths=0)
        sil = db['metrics']['sil']
        t = (f'{key} | {MODEL_LABELS[zkey]}\n'
             f'{db["n_found"]} clusters  noise={db["noise_pct"]:.0f}%')
        if not np.isnan(sil): t += f'  Sil={sil:.3f}'
        ax.set_title(t, fontsize=7, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(alpha=0.15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/dbscan_analysis.png', dpi=120, bbox_inches='tight')
plt.show(); print('✅ Saved: dbscan_analysis.png')


## 🔬 Step 19: Cluster Composition Heatmap

**Genre % within each K-Means cluster.** Split into two rows for readability.

In [ ]:
ROW1 = ['mlp', 'conv', 'hyb_conv', 'hyb_mlp', 'beta', 'cvae']
ROW2 = ['conv1d', 'ae', 'mm', 'pca', 'raw']

for key, res in all_results.items():
    if res is None: continue
    fig, axes = plt.subplots(2, max(len(ROW1), len(ROW2)),
                             figsize=(7 * max(len(ROW1), len(ROW2)), 14), squeeze=False)
    fig.suptitle(f'Cluster Composition — {res["name"]}\n(genre % within each K-Means cluster)',
                 fontsize=13, fontweight='bold')
    for row_idx, row_keys in enumerate([ROW1, ROW2]):
        for col_idx, zkey in enumerate(row_keys):
            ax     = axes[row_idx, col_idx]
            labels = res['cl'][zkey]['KMeans']['labels']
            df_tmp = pd.DataFrame({'cluster': labels, 'genre': res['y_labels']})
            ct     = pd.crosstab(df_tmp['cluster'], df_tmp['genre'])
            ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
            sns.heatmap(ct_pct, ax=ax, annot=True, fmt='.0f', cmap='YlOrRd',
                        linewidths=0.3, cbar_kws={'label': '%', 'shrink': 0.8},
                        annot_kws={'size': 6})
            ax.set_title(MODEL_LABELS[zkey], fontweight='bold', fontsize=10)
            ax.tick_params(axis='x', rotation=45, labelsize=7)
        for col_idx in range(len(row_keys), max(len(ROW1), len(ROW2))):
            axes[row_idx, col_idx].set_visible(False)
    plt.tight_layout()
    fname = f'{OUTPUT_DIR}/cluster_composition_{key.lower()}.png'
    plt.savefig(fname, dpi=130, bbox_inches='tight')
    plt.show(); print(f'✅ Saved: {fname}')


## 🇧🇩 Step 20: English vs Bangla Language Separation

Circle (●) = English  ·  Triangle (▲) = Bangla

In [ ]:
ROW1 = ['mlp', 'conv', 'hyb_conv', 'hyb_mlp', 'beta', 'cvae']
ROW2 = ['conv1d', 'ae', 'mm', 'pca', 'raw']
valid = [(k, v) for k, v in all_results.items() if v is not None]
n_cols = max(len(ROW1), len(ROW2))

fig, axes = plt.subplots(len(valid) * 2, n_cols,
                         figsize=(7 * n_cols, 5 * len(valid) * 2), squeeze=False)
fig.suptitle('English  vs  Bangla — UMAP Latent Space', fontsize=14, fontweight='bold')
for ds_idx, (key, res) in enumerate(valid):
    for row_offset, row_keys in enumerate([ROW1, ROW2]):
        row = ds_idx * 2 + row_offset
        for col, zkey in enumerate(row_keys):
            ax   = axes[row, col]
            Z2   = res['vis'][zkey]['umap']
            lang = res['lang_labels']
            for lng in ['English', 'Bangla']:
                mask = lang == lng
                if mask.any():
                    ax.scatter(Z2[mask, 0], Z2[mask, 1],
                               c=LANG_COL[lng], marker=LANG_MK[lng],
                               s=10, alpha=0.55, label=lng, linewidths=0)
            ax.set_title(f'{key} | {MODEL_LABELS[zkey]}', fontsize=9, fontweight='bold')
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(alpha=0.15)
            if ds_idx == 0 and row_offset == 0 and col == 0: ax.legend(fontsize=9)
        for col in range(len(row_keys), n_cols):
            axes[row, col].set_visible(False)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/language_separation.png', dpi=130, bbox_inches='tight')
plt.show(); print('✅ Saved: language_separation.png')


## 🔥 Step 21: Metrics Heatmap Dashboard (6 Metrics)

In [ ]:
ALGOS = ['KMeans', 'Agglomerative_Ward', 'Agglomerative_Complete', 'DBSCAN']
rows_list = []
for ds_key, res in all_results.items():
    if res is None: continue
    for zkey, zlab in MODEL_LABELS.items():
        if zkey not in res['cl']: continue
        for algo in ALGOS:
            if algo not in res['cl'][zkey]: continue
            m = res['cl'][zkey][algo]['metrics']
            rows_list.append({
                'Dataset': ds_key, 'Features': zlab, 'Algorithm': algo,
                'Silhouette':     round(m['sil'],   4) if not np.isnan(m['sil'])   else np.nan,
                'Davies-Bouldin': round(m['db'],    4) if not np.isnan(m['db'])    else np.nan,
                'Calinski-H':     round(m['ch'],    1) if not np.isnan(m['ch'])    else np.nan,
                'ARI':            round(m['ari'],   4) if not np.isnan(m['ari'])   else np.nan,
                'NMI':            round(m['nmi'],   4) if not np.isnan(m['nmi'])   else np.nan,
                'Purity':         round(m['purity'],4) if not np.isnan(m['purity']) else np.nan,
            })

df_all = pd.DataFrame(rows_list)
pd.set_option('display.max_rows', 200); pd.set_option('display.width', 200)
print('=' * 110)
print('  FULL METRICS TABLE  |  Sil↑  DB↓  CH↑  ARI↑  NMI↑  Purity↑')
print('=' * 110)
print(df_all.to_string(index=False))
df_all.to_csv(f'{OUTPUT_DIR}/full_metrics.csv', index=False)
print('\n✅ Saved: full_metrics.csv')

feat_order  = list(MODEL_LABELS.values())
metrics_cfg = [
    ('Silhouette',     '↑ higher', 'Blues'),
    ('Davies-Bouldin', '↓ lower',  'Reds_r'),
    ('Calinski-H',     '↑ higher', 'Greens'),
    ('ARI',            '↑ higher', 'Purples'),
    ('NMI',            '↑ higher', 'Oranges'),
    ('Purity',         '↑ higher', 'YlOrBr'),
]
fig, axes = plt.subplots(2, 3, figsize=(28, 16), squeeze=False)
fig.suptitle('Clustering Quality Heatmap — All 6 Metrics\nRows=Dataset+Algorithm | Cols=Feature Space',
             fontsize=14, fontweight='bold')
for ax, (metric, note, cmap) in zip(axes.flat, metrics_cfg):
    if metric not in df_all.columns: ax.set_visible(False); continue
    pivot = df_all.pivot_table(index=['Dataset','Algorithm'], columns='Features',
                               values=metric, aggfunc='mean')
    cols = [c for c in feat_order if c in pivot.columns]
    sns.heatmap(pivot[cols].astype(float), ax=ax, annot=True, fmt='.3f',
                cmap=cmap, linewidths=0.4, linecolor='white', cbar_kws={'shrink': 0.8})
    ax.set_title(f'{metric}  ({note})', fontweight='bold', fontsize=11)
    ax.set_xlabel('Feature Space')
    ax.tick_params(axis='x', rotation=20, labelsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/metrics_heatmap.png', dpi=150, bbox_inches='tight')
plt.show(); print('✅ Saved: metrics_heatmap.png')


## 📊 Step 22: Best Metrics Bar Charts

In [ ]:
feat_cols = list(MODEL_LABELS.values())
datasets  = [k for k, v in all_results.items() if v is not None]
x = np.arange(len(datasets))
w = 0.07   # narrower — 11 models

summary = []
for ds_key in datasets:
    res = all_results[ds_key]
    for zkey, zlab in MODEL_LABELS.items():
        if zkey not in res['cl']: continue
        def _b(m, fn):
            v = [res['cl'][zkey][a]['metrics'][m]
                 for a in ['KMeans', 'Agglomerative_Ward']
                 if a in res['cl'][zkey] and not np.isnan(res['cl'][zkey][a]['metrics'][m])]
            return fn(v) if v else np.nan
        summary.append({'Dataset': ds_key, 'Features': zlab,
                         'Best Sil': _b('sil', max), 'Best NMI': _b('nmi', max),
                         'Best ARI': _b('ari', max), 'Best Purity': _b('purity', max)})
df_sum = pd.DataFrame(summary)

fig, axes = plt.subplots(1, 4, figsize=(32, 7))
fig.suptitle('Best Score per Feature Space (K-Means & Agglom-Ward)', fontsize=13, fontweight='bold')
for ax, (metric, ylabel) in zip(axes, [
    ('Best Sil',    'Silhouette (↑)'),
    ('Best NMI',    'NMI (↑)'),
    ('Best ARI',    'ARI (↑)'),
    ('Best Purity', 'Purity (↑)'),
]):
    for fi, feat in enumerate(feat_cols):
        subset = df_sum[df_sum.Features == feat]
        vals   = [subset[subset.Dataset == d][metric].values[0] if len(subset[subset.Dataset == d]) > 0 else np.nan
                  for d in datasets]
        offset = fi * w - (len(feat_cols) / 2) * w
        bars   = ax.bar(x + offset, vals, w, label=feat,
                        color=COLORS_M.get(feat, '#888888'), alpha=0.88)
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                        f'{v:.3f}', ha='center', va='bottom', fontsize=5, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(datasets, fontsize=11)
    ax.set_ylabel(ylabel); ax.set_title(ylabel, fontweight='bold')
    ax.legend(fontsize=7, ncol=2); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/best_metrics_bar.png', dpi=150, bbox_inches='tight')
plt.show(); print('✅ Saved: best_metrics_bar.png')


## 📉 Step 23: Training Loss Curves

In [ ]:
MODEL_COLORS_TRAIN = {
    'mlp':      '#1565C0', 'conv':     '#6A1B9A',
    'hyb_conv': '#2E7D32', 'hyb_mlp':  '#E65100',
    'beta':     '#AD1457', 'cvae':     '#00838F',
    'conv1d':   '#558B2F', 'ae':       '#FF8F00',
    'mm':       '#00695C',
}
valid  = [(k, v) for k, v in all_results.items() if v is not None]
n_cols = len(valid)
fig, axes = plt.subplots(1, n_cols, figsize=(8 * n_cols, 5), squeeze=False)
fig.suptitle('VAE Training Loss Curves per Dataset', fontsize=13, fontweight='bold')
for ax, (key, res) in zip(axes[0], valid):
    for mkey, color in MODEL_COLORS_TRAIN.items():
        hist = res['hist'].get(mkey)
        loss = res['loss'].get(mkey)
        if hist is None or loss is None: continue
        train_losses = [h[0] for h in hist]
        ax.plot(train_losses, color=color, linewidth=2,
                label=f'{MODEL_LABELS.get(mkey, mkey)} ({loss:.4f})')
    ax.set_title(key, fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel('Train Loss')
    ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show(); print('✅ Saved: training_curves.png')


## 🔬 Step 24: VAE vs PCA Baseline — Δ Analysis

**Positive bar** = model beats PCA baseline on that metric.

In [ ]:
vae_keys_delta = [(k, v) for k, v in MODEL_LABELS.items() if k not in ('pca', 'raw')]
valid    = [(k, v) for k, v in all_results.items() if v is not None]
datasets = [k for k, _ in valid]
x        = np.arange(len(datasets))
w        = 0.07

fig, axes = plt.subplots(1, 2, figsize=(26, 6), squeeze=False)
fig.suptitle('VAE vs PCA Baseline — Δ Silhouette & Δ NMI (K-Means)\n'
             'Positive = better than PCA | Negative = worse',
             fontsize=13, fontweight='bold')

for ax, metric in zip(axes[0], ['sil', 'nmi']):
    ylabel = f'Δ {metric.upper()} (model − PCA)'
    for fi, (zkey, zlabel) in enumerate(vae_keys_delta):
        deltas = []
        for _, res in valid:
            vae_m = res['cl'].get(zkey, {}).get('KMeans', {}).get('metrics', {})
            pca_m = res['cl'].get('pca',  {}).get('KMeans', {}).get('metrics', {})
            v_val = vae_m.get(metric, np.nan)
            p_val = pca_m.get(metric, np.nan)
            deltas.append(v_val - p_val if not (np.isnan(v_val) or np.isnan(p_val)) else np.nan)
        offset = fi * w - (len(vae_keys_delta) / 2) * w
        bars   = ax.bar(x + offset, deltas, w, label=zlabel,
                        color=COLORS_M.get(zlabel, '#888888'), alpha=0.85)
        for bar, v in zip(bars, deltas):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + (0.002 if v >= 0 else -0.012),
                        f'{v:+.3f}', ha='center', va='bottom', fontsize=5, fontweight='bold')
    ax.axhline(0, color='black', linewidth=1, linestyle='--', label='PCA baseline')
    ax.set_xticks(x); ax.set_xticklabels(datasets, fontsize=11)
    ax.set_ylabel(ylabel); ax.set_title(ylabel, fontweight='bold')
    ax.legend(fontsize=7, ncol=2); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/vae_vs_baseline.png', dpi=150, bbox_inches='tight')
plt.show(); print('✅ Saved: vae_vs_baseline.png')


## 🎛️ Step 25: Beta-VAE Disentanglement Analysis

Histograms of latent dimensions — MLP-VAE vs Beta-VAE vs CVAE.

In [ ]:
for key, res in all_results.items():
    if res is None: continue
    Z_mlp  = res['Z']['mlp']
    Z_beta = res['Z']['beta']
    Z_cvae = res['Z']['cvae']
    y_true = res['y_true']
    n_cl   = res['n_class']
    PAL    = plt.colormaps['tab20'].resampled(n_cl)
    n_show = min(8, LATENT_DIM)
    best_beta = res.get('best_beta', BETA_VAE_B)

    models_to_show = [
        (Z_mlp,  'MLP-VAE (baseline)'),
        (Z_beta, f'Beta-VAE (β={best_beta:.1f}, best)'),
        (Z_cvae, 'CVAE (zero-condition)'),
    ]
    fig, axes = plt.subplots(len(models_to_show), n_show,
                             figsize=(n_show * 3.5, 4 * len(models_to_show)), squeeze=False)
    fig.suptitle(f'Latent Dimension Distributions — Disentanglement Analysis\n{res["name"]}',
                 fontsize=13, fontweight='bold')
    for row, (Z, title) in enumerate(models_to_show):
        for di in range(n_show):
            ax = axes[row, di]
            for gi in range(n_cl):
                vals = Z[y_true == gi, di]
                if len(vals) > 0:
                    ax.hist(vals, bins=20, alpha=0.55, color=PAL(gi), density=True,
                            label=res['le'].classes_[gi] if di == 0 else None)
            ax.set_title(f'{title}\ndim {di}' if di == 0 else f'dim {di}', fontsize=8)
            ax.set_yticks([]); ax.grid(alpha=0.2)
        axes[row, 0].legend(fontsize=6, loc='upper right', title='genre', title_fontsize=6)
    plt.tight_layout()
    fname = f'{OUTPUT_DIR}/disentangle_{key.lower()}.png'
    plt.savefig(fname, dpi=130, bbox_inches='tight')
    plt.show(); print(f'✅ Saved: {fname}')


## 🔭 Step 26: Beta-VAE Latent Traversal

What does each latent dimension encode?

In [ ]:
_traversal_key = next(
    (k for k, v in all_results.items()
     if v is not None and v['models'].get('beta') is not None), None)
if _traversal_key is None:
    print('No valid Beta-VAE model — skipping latent traversal.')
else:
    res   = all_results[_traversal_key]
    model = res['models']['beta'].eval()
    Z_mean = torch.FloatTensor(res['Z']['beta']).mean(0)
    n_dims  = 8; n_steps = 10
    z_range = np.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(2*n_steps, 2*n_dims), squeeze=False)
    for di in range(n_dims):
        for ti, val in enumerate(z_range):
            z = Z_mean.clone(); z[di] = val
            with torch.no_grad():
                recon = model.decode(z.unsqueeze(0).to(DEVICE)).cpu().numpy().flatten()
            ax = axes[di, ti]
            ax.plot(recon, lw=0.8, color='#1565C0')
            ax.set_xticks([]); ax.set_yticks([])
            if ti == 0: ax.set_ylabel(f'dim {di}', fontsize=7)
            if di == 0: ax.set_title(f'z={val:+.1f}', fontsize=7)
    plt.suptitle(f'Beta-VAE Latent Traversal — {_traversal_key}  (β={res["best_beta"]:.1f})',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/latent_traversal.png', dpi=120, bbox_inches='tight')
    plt.show(); print(f'✅ Saved: latent_traversal.png  (dataset={_traversal_key})')


## 🔁 Step 27: Reconstruction Examples

Original vs reconstructed feature vectors — MLP-VAE, Beta-VAE, CVAE.

In [ ]:
rng = np.random.default_rng(SEED)
for key, res in all_results.items():
    if res is None: continue
    X_sc   = res['X_sc']
    models = res['models']
    n_show = 5
    idx    = rng.choice(len(X_sc), min(n_show, len(X_sc)), replace=False)
    fig, axes = plt.subplots(len(idx), 3, figsize=(18, len(idx) * 2.8), squeeze=False)
    fig.suptitle(f'Reconstruction Examples — {res["name"]}', fontsize=13, fontweight='bold')
    for row, i in enumerate(idx):
        x_orig = torch.FloatTensor(X_sc[i:i+1]).to(DEVICE)
        genre  = res['y_labels'][i]
        for col, (mkey, model) in enumerate([
            ('MLP-VAE', models['mlp']),
            ('Beta-VAE', models['beta']),
            ('CVAE',     models['cvae']),
        ]):
            model.eval()
            with torch.no_grad():
                if mkey == 'CVAE':
                    n_cl = res['n_class']
                    c    = torch.zeros(1, n_cl, device=DEVICE)
                    c[0, res['le'].transform([genre])[0]] = 1.0
                    recon, _, _, _ = model(x_orig, c)
                else:
                    recon, _, _, _ = model(x_orig)
            orig_np  = x_orig.cpu().numpy().flatten()
            recon_np = recon.cpu().numpy().flatten()
            show_dim = min(AUDIO_FEAT_DIM, len(orig_np))
            ax = axes[row, col]
            ax.plot(orig_np[:show_dim],  color='#1565C0', lw=1.5, label='Original' if row==0 else None)
            ax.plot(recon_np[:show_dim], color='#FF5722', lw=1.5, linestyle='--', label='Recon' if row==0 else None)
            mse = float(np.mean((orig_np[:show_dim] - recon_np[:show_dim]) ** 2))
            ax.set_title(f'{mkey} | {genre[:12]} | MSE={mse:.4f}', fontsize=7)
            ax.set_yticks([]); ax.grid(alpha=0.2)
            if row == 0: ax.legend(fontsize=7)
    plt.tight_layout()
    fname = f'{OUTPUT_DIR}/reconstruction_{key.lower()}.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.show(); print(f'✅ Saved: {fname}')


## 📋 Step 28: Quantitative Analysis & Interpretation

In [ ]:
print('=' * 80)
print('  ANALYSIS: All VAE Variants vs PCA Baseline (K-Means, Silhouette + NMI)')
print('=' * 80)
for ds_key, res in all_results.items():
    if res is None: continue
    print(f'\n  Dataset: {ds_key}')
    print(f'  {"":-<76}')
    pca_sil = res['cl']['pca']['KMeans']['metrics']['sil']
    pca_nmi = res['cl']['pca']['KMeans']['metrics']['nmi']
    for zkey, zlab in MODEL_LABELS.items():
        if zkey in ('pca', 'raw') or zkey not in res['cl']: continue
        m       = res['cl'][zkey]['KMeans']['metrics']
        vae_sil = m['sil']; vae_nmi = m['nmi']
        if np.isnan(vae_sil) or np.isnan(pca_sil):
            print(f'  {ds_key:<12} | {zlab:<20} | Sil: NaN — skipped'); continue
        d_sil = vae_sil - pca_sil; d_nmi = vae_nmi - pca_nmi
        pct   = d_sil / abs(pca_sil) * 100 if pca_sil != 0 else 0
        v     = ('BETTER ✅' if d_sil > 0.005 else 'WORSE  ❌' if d_sil < -0.005 else 'SIMILAR ~')
        print(f'  {ds_key:<12} | {zlab:<20} | '
              f'Sil: {vae_sil:.4f} vs PCA {pca_sil:.4f} '
              f'Δ={d_sil:+.4f}({pct:+.1f}%)  NMI Δ={d_nmi:+.4f}  {v}')

print()
print('INTERPRETATION')
print('─' * 70)
print('✅ BETTER  → Non-linear encoder captures manifold structure PCA cannot.')
print('   When: high-dim data, complex genre boundaries, entangled audio features.')
print()
print('❌ WORSE   → Small dataset (VAE overfits), very low-dim data, β too high.')
print()
print('Conv2D-VAE  : captures local time-frequency correlations (delta-stacked MFCC).')
print('HybridConvVAE: end-to-end Conv2D + lyric fusion — strongest when real lyrics available.')
print('Beta-VAE    : disentangled latent → individual dims align with audio factors.')
print('CVAE        : genre-aware latent — useful for conditional generation.')
print('MultiModalVAE: joint audio+lyric encoder, best when lyrics are informative.')
print()
print('NMI: symmetric, corrects for cluster size. ARI: corrects for random labelling.')
print('Purity: fraction of samples in majority-genre clusters — easy to interpret.')


## 🆚 Step 28b: Head-to-Head Comparison — VAE-Based vs PCA+K-Means vs AE+K-Means vs Direct Spectral Clustering

This step runs a **fair, side-by-side comparison** of four clustering paradigms:

| Paradigm | Method | Notes |
|----------|--------|-------|
| 🔵 VAE-Based | Best VAE variant latent space + K-Means | Picks best-performing VAE per dataset |
| 🔴 PCA + K-Means | PCA (32 components) → K-Means | Classic linear baseline |
| 🟠 Autoencoder + K-Means | Deterministic AE latent → K-Means | Non-linear, no KL regularisation |
| 🟢 Direct Spectral | Raw 65-dim features → K-Means | No dimensionality reduction |

All four are evaluated on the **same 6 metrics** (Silhouette ↑, DB ↓, CH ↑, ARI ↑, NMI ↑, Purity ↑). A bar-chart, radar chart, and ranked summary table are produced for each dataset.

In [ ]:
# =============================================================================
# Step 28b -- Head-to-Head Comparison:
#   VAE-Based Clustering  vs  PCA + K-Means  vs  AE + K-Means  vs
#   Direct Spectral Feature Clustering
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

# ---- helpers -----------------------------------------------------------------
def _get_kmeans_metrics(res, zkey):
    # Return KMeans metrics dict for a feature-space key, or None
    try:
        return res['cl'][zkey]['KMeans']['metrics']
    except (KeyError, TypeError):
        return None

def _best_vae_key(res):
    # Return the VAE key with the highest KMeans Silhouette score
    VAE_KEYS = ['mlp', 'conv', 'hyb_conv', 'hyb_mlp', 'beta', 'cvae', 'conv1d', 'mm']
    best_key, best_sil = None, -np.inf
    for k in VAE_KEYS:
        m = _get_kmeans_metrics(res, k)
        if m is not None and not np.isnan(m.get('sil', np.nan)):
            if m['sil'] > best_sil:
                best_sil, best_key = m['sil'], k
    return best_key, best_sil

METRICS_INFO = [
    ('sil',    'Silhouette',     True),
    ('db',     'Davies-Bouldin', False),
    ('ch',     'Calinski-H',     True),
    ('ari',    'ARI',            True),
    ('nmi',    'NMI',            True),
    ('purity', 'Purity',         True),
]

PARADIGMS = [
    ('vae_best',  'Best-VAE',             '#1565C0'),
    ('pca',       'PCA + K-Means',         '#B71C1C'),
    ('ae',        'Autoencoder + K-Means', '#FF8F00'),
    ('raw',       'Direct Spectral',       '#2E7D32'),
]

METRIC_LABELS = {
    'sil':    'Silhouette (up)',
    'db':     'Davies-Bouldin (dn)',
    'ch':     'Calinski-H (up)',
    'ari':    'ARI (up)',
    'nmi':    'NMI (up)',
    'purity': 'Purity (up)',
}

# ---- collect results ---------------------------------------------------------
comparison_rows = []
vae_best_info   = {}

print('=' * 90)
print('  PARADIGM COMPARISON  |  K-Means  |  All 6 Metrics')
print('=' * 90)

for ds_key, res in all_results.items():
    if res is None:
        continue

    best_vae_k, _ = _best_vae_key(res)
    vae_best_info[ds_key] = (best_vae_k, MODEL_LABELS.get(best_vae_k, str(best_vae_k)))

    for p_key, p_label, _ in PARADIGMS:
        if p_key == 'vae_best':
            m = _get_kmeans_metrics(res, best_vae_k) if best_vae_k else None
        else:
            m = _get_kmeans_metrics(res, p_key)

        row = {'Dataset': ds_key, 'Paradigm': p_label}
        for mk, mlabel, _ in METRICS_INFO:
            row[METRIC_LABELS[mk]] = m.get(mk, np.nan) if m else np.nan
        comparison_rows.append(row)

df_cmp = pd.DataFrame(comparison_rows)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 180)
print(df_cmp.to_string(index=False))

print()
print('  Best VAE variant selected per dataset:')
for ds_key, (bk, bl) in vae_best_info.items():
    print(f'    {ds_key:<12} -> {bl}')

# ---- bar chart -- one subplot per metric ------------------------------------
datasets  = [k for k, v in all_results.items() if v is not None]
n_ds      = len(datasets)
n_para    = len(PARADIGMS)
w         = 0.18
x         = np.arange(n_ds)

fig, axes = plt.subplots(2, 3, figsize=(28, 14), squeeze=False)
fig.suptitle(
    'Head-to-Head Clustering Paradigm Comparison\n'
    'Blue=Best-VAE  |  Red=PCA+KMeans  |  Orange=AE+KMeans  |  Green=Direct Spectral',
    fontsize=14, fontweight='bold'
)

metric_col_order = [METRIC_LABELS[mk] for mk, _, _ in METRICS_INFO]
higher_map = {METRIC_LABELS[mk]: hb for mk, _, hb in METRICS_INFO}

for ax, col in zip(axes.flat, metric_col_order):
    for fi, (p_key, p_label, color) in enumerate(PARADIGMS):
        vals = []
        for ds_key in datasets:
            row = df_cmp[(df_cmp['Dataset'] == ds_key) & (df_cmp['Paradigm'] == p_label)]
            vals.append(float(row[col].values[0]) if len(row) > 0 else np.nan)

        offset = (fi - n_para / 2 + 0.5) * w
        bars   = ax.bar(x + offset, vals, w, label=p_label, color=color, alpha=0.85,
                        edgecolor='white', linewidth=0.5)
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + abs(bar.get_height()) * 0.015,
                        f'{v:.3f}', ha='center', va='bottom', fontsize=6.5,
                        fontweight='bold', rotation=45)

    arrow = 'higher is better' if higher_map[col] else 'lower is better'
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, fontsize=11)
    ax.set_ylabel(col, fontsize=10)
    ax.set_title(f'{col}  ({arrow})', fontweight='bold', fontsize=11)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/paradigm_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: paradigm_comparison_bar.png')

# ---- ranked summary table ---------------------------------------------------
print()
print('=' * 90)
print('  RANKED SUMMARY (rank 1 = best per metric per dataset)')
print('=' * 90)

rank_rows = []
for ds_key in datasets:
    sub = df_cmp[df_cmp['Dataset'] == ds_key].copy()
    for col, hb in higher_map.items():
        vals = sub[col].values.astype(float)
        order  = np.argsort(vals * (-1 if hb else 1))
        ranks  = np.empty_like(order)
        ranks[order] = np.arange(1, n_para + 1)
        for i, (_, p_label, _) in enumerate(PARADIGMS):
            rank_rows.append({
                'Dataset':  ds_key,
                'Paradigm': p_label,
                'Metric':   col,
                'Value':    vals[i],
                'Rank':     ranks[i],
            })

df_rank = pd.DataFrame(rank_rows)
avg_rank = (df_rank.groupby(['Dataset', 'Paradigm'])['Rank']
                   .mean().reset_index()
                   .rename(columns={'Rank': 'Avg Rank (lower=better)'}))
avg_rank['Avg Rank (lower=better)'] = avg_rank['Avg Rank (lower=better)'].round(2)
pivot_rank = avg_rank.pivot(index='Dataset', columns='Paradigm',
                            values='Avg Rank (lower=better)')
print(pivot_rank.to_string())

print()
print('  Winner (lowest avg rank) per dataset:')
for ds_key in datasets:
    row = pivot_rank.loc[ds_key]
    winner = row.idxmin()
    print(f'    {ds_key:<12} -> {winner}  (avg rank {row.min():.2f})')

# ---- radar / spider chart -- mean across all datasets -----------------------
try:
    radar_metric_cols = [METRIC_LABELS[mk] for mk in ['sil', 'ari', 'nmi', 'purity', 'ch', 'db']]
    N = len(radar_metric_cols)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    fig_r, ax_r = plt.subplots(1, 1, figsize=(8, 8), subplot_kw=dict(polar=True))
    fig_r.suptitle(
        'Paradigm Radar -- Mean Score Across All Datasets\n'
        '(all metrics normalised to [0,1]; Davies-Bouldin inverted)',
        fontsize=12, fontweight='bold'
    )

    radar_data = {}
    for _, p_label, _ in PARADIGMS:
        sub    = df_cmp[df_cmp['Paradigm'] == p_label]
        scores = [float(sub[col].mean()) for col in radar_metric_cols]
        radar_data[p_label] = scores

    all_sc  = np.array(list(radar_data.values()))
    col_min = all_sc.min(axis=0)
    col_max = all_sc.max(axis=0)
    col_rng = np.where(col_max - col_min < 1e-9, 1.0, col_max - col_min)

    db_idx = radar_metric_cols.index(METRIC_LABELS['db'])

    for (_, p_label, color), raw_sc in zip(PARADIGMS, all_sc):
        norm_sc = (raw_sc - col_min) / col_rng
        norm_sc[db_idx] = 1.0 - norm_sc[db_idx]   # invert DB
        values = norm_sc.tolist() + [norm_sc[0]]
        ax_r.plot(angles, values, 'o-', linewidth=2, color=color, label=p_label)
        ax_r.fill(angles, values, alpha=0.12, color=color)

    radar_tick_labels = ['Silhouette', 'ARI', 'NMI', 'Purity', 'Calinski-H', 'DB (inv)']
    ax_r.set_xticks(angles[:-1])
    ax_r.set_xticklabels(radar_tick_labels, size=10)
    ax_r.set_ylim(0, 1)
    ax_r.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax_r.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], size=7, color='grey')
    ax_r.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
    ax_r.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.6)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/paradigm_radar.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: paradigm_radar.png')
except Exception as e_radar:
    print(f'  [Radar chart skipped: {e_radar}]')

# ---- export CSV -------------------------------------------------------------
df_cmp.to_csv(f'{OUTPUT_DIR}/paradigm_comparison.csv', index=False)
print('Saved: paradigm_comparison.csv')

print()
print('INTERPRETATION')
print('-' * 70)
print('Blue  Best-VAE       : non-linear latent + KL regularisation -> smooth,')
print('                       clusterable space. Wins on complex genre boundaries.')
print('Red   PCA+K-Means    : fast, interpretable linear baseline. Good on clean')
print('                       low-dim data; loses when manifold is non-linear.')
print('Orange AE+K-Means    : non-linear compression but NO KL -> latent may')
print('                       collapse or over-fit. Typically sits between PCA/VAE.')
print('Green  Direct Spect  : K-Means on raw 65-dim features. Can beat all when')
print('                       features are already well-separated; noisy otherwise.')


## 📋 Step 29: Final Summary Report

In [ ]:
SEP = '=' * 80
print(SEP)
print('  ADVANCED MULTI-MODAL VAE CLUSTERING — FINAL REPORT  (Combined Task)')
print('  3 Datasets: GTZAN · BanglaGITI · BMGCD')
print('  11 Feature Spaces × 4 Algorithms × 6 Metrics | No Synthetic Audio Data')
print(SEP)

def _f(v):
    return f'{v:.4f}' if isinstance(v, float) and not np.isnan(v) else '  N/A '

for ds_key, res in all_results.items():
    if res is None: continue
    print(f'\n  {res["name"]}  |  {len(res["y_labels"])} samples  {res["n_class"]} genres')
    print(f'  {"Features":<20} {"Algorithm":<24} {"Sil":>7} {"DB(↓)":>7} {"CH":>9} {"ARI":>7} {"NMI":>7} {"Purity":>7}')
    print('  ' + '-' * 87)
    for zkey, zlab in MODEL_LABELS.items():
        if zkey not in res['cl']: continue
        for algo in ['KMeans', 'Agglomerative_Ward', 'Agglomerative_Complete', 'DBSCAN']:
            if algo not in res['cl'][zkey]: continue
            m = res['cl'][zkey][algo]['metrics']
            print(f'  {zlab:<20} {algo:<24} {_f(m["sil"]):>7} {_f(m["db"]):>7} '
                  f'{_f(m["ch"]):>9} {_f(m["ari"]):>7} {_f(m["nmi"]):>7} {_f(m["purity"]):>7}')

print()
print(SEP)
print('  LYRICS COVERAGE SUMMARY')
print(SEP)
for ds_key, res in all_results.items():
    if res is None: continue
    has_real = res.get('has_real_lyrics', np.array([]))
    n_real   = int(np.sum(has_real))
    n_total  = len(res['y_labels'])
    pct      = 100 * n_real / max(n_total, 1)
    print(f'  {ds_key:<12}: {n_real:>4}/{n_total} real lyrics ({pct:.1f}%) '
          f'| {n_total - n_real} neutral fallback')
print()
print('  Note: GTZAN lyrics are 100% genre-seed based (numeric filenames = no title).')
print('  BanglaGITI/BMGCD: real lyrics scraped where available (gaanesuno.com).')
print('  MultiModalVAE uses joint audio+lyric encoder — strongest with real lyrics.')


## 📥 Step 30: Download All Results

In [ ]:
import shutil
shutil.make_archive('/content/vae_combined_results', 'zip', OUTPUT_DIR)
from google.colab import files
files.download('/content/vae_combined_results.zip')
print('✅ Download started!')
